# Pipeline prediksi suhu ruang server dalam satu notebook

Notebook ini membawa kode, konfigurasi, dan CSV sumber yang sama dengan versi modular di GitHub, commit 3b07a61. Unduh atau salin berkas ini saja, buka dengan Jupyter, pilih tahap, lalu jalankan **Run All**. Notebook tidak perlu mengunduh kode maupun dataset.

- **praproses** (bawaan) memeriksa data dan membentuk sampel latih, validasi, serta uji. GPU tidak dibutuhkan.
- **pembanding** menambahkan prediksi Naive, Seasonal Naive, Moving Average, dan ARIMA.
- **lengkap** menjalankan praproses, pemeriksaan singkat GPU, estimasi waktu, tuning, pelatihan final LSTM dan GRU, pembanding, serta evaluasi. Gunakan kernel Python 3.12 di Linux atau WSL dengan GPU yang terdeteksi oleh TensorFlow. Jika estimasi melebihi batas konfigurasi, controller menghentikan run untuk ditinjau sebelum tuning.
- **uji** menjalankan tes kode dengan data sintetis.

CSV sumber berisi 76.979 pembacaan. Praproses memilih periode penelitian 20 Agustus–30 September 2026 (WIB) dari CSV tersebut. Model terlatih dan keluaran dibuat pada saat eksekusi. Evaluasi mengikuti konfigurasi versi modular. Analisis peringatan historis memiliki protokol terpisah dari alur ini.

Setiap eksekusi membuat subfolder baru di bawah folder keluaran, dengan salinan kode dan CSV untuk menelusuri hasil. Dependensi Python perlu tersedia pada kernel. Opsi pemasangan di bawah menggunakan akses internet untuk paket Python.

In [ ]:
TAHAP = "praproses"  # "praproses", "pembanding", "lengkap", atau "uji"
PASANG_DEPENDENSI = False  # True jika paket belum terpasang dan internet tersedia
FOLDER_KELUARAN = "hasil_eksperimen_notebook"

## Alur eksekusi

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import base64
import csv
import gzip
import hashlib
import importlib.util
import json
import os
import subprocess
import sys

KODE_PIPELINE = {}

def jalankan(args, direktori, log, tampilkan=True):
    """Jalankan satu tahap dengan interpreter kernel dan simpan keluaran lengkap."""
    print(f"Menjalankan {args[0]}. Log: {log}", flush=True)
    with log.open("w", encoding="utf-8") as berkas:
        status = subprocess.run(
            [sys.executable, "-u", *args], cwd=direktori,
            stdout=berkas, stderr=subprocess.STDOUT,
        )
    if tampilkan or status.returncode:
        cuplikan = log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
        print("\n".join(cuplikan))
    if status.returncode:
        raise RuntimeError(f"Tahap gagal ({status.returncode}). Lihat {log}.")

def periksa_gpu(direktori, log):
    if os.name != "posix":
        raise RuntimeError("Buka notebook dengan kernel Linux/WSL untuk tahap lengkap.")
    jalankan(
        [
            "-c",
            "import tensorflow as tf; "
            "gpu = tf.config.list_physical_devices('GPU'); "
            "print(gpu); "
            "assert gpu, 'TensorFlow tidak mendeteksi GPU.'",
        ],
        direktori,
        log,
    )

def mulai():
    tahap = TAHAP.strip().lower()
    if tahap not in {"praproses", "pembanding", "lengkap", "uji"}:
        raise ValueError("Pilih tahap praproses, pembanding, lengkap, atau uji.")

    data = gzip.decompress(base64.b64decode(DATA_CSV_B64))
    checksum = hashlib.sha256(data).hexdigest()
    if checksum != DATA_CSV_SHA256:
        raise RuntimeError("Checksum dataset di dalam notebook tidak cocok.")

    tujuan = Path(FOLDER_KELUARAN).expanduser().resolve()
    tujuan.mkdir(parents=True, exist_ok=True)
    run = Path(mkdtemp(prefix=f"{tahap}-", dir=tujuan))
    sumber = run / "_sumber_pipeline"
    sumber.mkdir()
    for nama, isi in KODE_PIPELINE.items():
        target = sumber / nama
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(isi.encode("utf-8"))
    csv_sumber = sumber / "01_dataset_sumber" / "sensor_raw.csv"
    csv_sumber.parent.mkdir()
    csv_sumber.write_bytes(data)
    (run / "sumber_notebook.json").write_text(
        json.dumps(
            {"commit_sumber": COMMIT_SUMBER, "sha256_csv": checksum, "tahap": tahap},
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )
    print("Folder run:", run)

    paket = {"numpy": "numpy", "pandas": "pandas", "yaml": "PyYAML"}
    if tahap in {"pembanding", "lengkap", "uji"}:
        paket.update({"scipy": "SciPy", "statsmodels": "statsmodels"})
    if tahap == "lengkap":
        paket["tensorflow"] = "TensorFlow"
    if PASANG_DEPENDENSI:
        jalankan(
            ["-m", "pip", "install", "-r", "requirements.txt"],
            sumber,
            run / "dependensi.log",
        )
        importlib.invalidate_caches()
    hilang = [nama for modul, nama in paket.items() if importlib.util.find_spec(modul) is None]
    if hilang:
        raise RuntimeError(
            "Paket belum tersedia pada kernel: " + ", ".join(hilang)
            + ". Pasang paket atau pilih PASANG_DEPENDENSI=True."
        )

    if tahap == "uji":
        jalankan(
            ["-m", "unittest", "discover", "-s", "pemeriksaan_kode", "-v"],
            sumber,
            run / "uji.log",
        )
        return run

    opsi = ["--config", "config.yaml", "--run-dir", str(run)]
    if tahap == "lengkap":
        periksa_gpu(sumber, run / "gpu.log")
        jalankan(["04_training/42_run_pipeline.py", "worker", *opsi], sumber, run / "training.log")
        status = json.loads((run / "logs" / "pipeline_status.json").read_text())["status"]
        if status == "WAITING_DURATION_REVIEW":
            print("Benchmark melewati batas durasi. Tinjau logs/benchmark.json sebelum melanjutkan.")
            return run
        if status != "FINAL_TRAINING_COMPLETED":
            raise RuntimeError(f"Controller belum menuntaskan pelatihan final: {status}")
    else:
        jalankan(["03_praproses/02_prepare.py", *opsi], sumber, run / "praproses.log")

    if tahap in {"pembanding", "lengkap"}:
        jalankan(["05_baseline/03_baselines.py", *opsi], sumber, run / "pembanding.log")
    if tahap == "lengkap":
        jalankan(["06_evaluasi/05_evaluate.py", *opsi], sumber, run / "evaluasi.log")

    hasil = run / "results"
    with (hasil / "sequence_counts.csv").open(newline="", encoding="utf-8") as berkas:
        jumlah = {
            (row["horizon_minutes"], row["split"]): row["samples"]
            for row in csv.DictReader(berkas)
        }
    print("\nJumlah sampel sebelum irisan evaluasi")
    print("Horizon (menit) | Latih | Validasi | Uji")
    for horizon in ("60", "120", "180"):
        print(" | ".join([horizon, *(jumlah[horizon, split] for split in ("train", "val", "test"))]))
    print(f"\nTahap {tahap} selesai. Keluaran: {hasil}")
    return run

## Konfigurasi dan kode bawaan

Sel berikut memuat sumber yang digunakan oleh notebook. Saat dijalankan, sumber disalin ke folder run agar setiap tahap memakai modul dan konfigurasi yang sama.

In [ ]:
# config.yaml
KODE_PIPELINE['config.yaml'] = '''project:
  status: pc_final_42_days_train_only_20261013
  timezone: Asia/Jakarta
  run_root: hasil_eksperimen
data:
  csv_path: 01_dataset_sumber/sensor_raw.csv
  kolom_waktu: waktu_wib
  kolom_target: temperature
  kolom_kelembapan: humidity
  mulai: '2026-08-20 00:00:00+07:00'
  akhir_eksklusif: '2026-10-01 00:00:00+07:00'
  toleransi_ujung_menit: 15
  artefak_ranges:
  - mulai_eksklusif: '2026-08-26 09:35:15.777+07:00'
    selesai_eksklusif: '2026-08-26 13:18:40.717+07:00'
  batas_operasi_suhu:
  - -10.0
  - 60.0
  kandidat_harus_lengkap: true
preprocessing:
  resample: 5min
  agregasi: mean
  max_gap_interpolasi_menit: 15
  lookback: 144
  horizons:
  - 12
  - 24
  - 36
  features:
  - temperature
  - humidity
  - jam_sin
  - jam_cos
  - hari_sin
  - hari_cos
  - akhir_pekan
  split:
    train:
    - '2026-08-20 00:00:00+07:00'
    - '2026-09-17 00:00:00+07:00'
    val:
    - '2026-09-17 00:00:00+07:00'
    - '2026-09-24 00:00:00+07:00'
    test:
    - '2026-09-24 00:00:00+07:00'
    - '2026-10-01 00:00:00+07:00'
  scaler_fit: train_only
model:
  architectures:
  - lstm
  - gru
  grid:
    units:
    - 32
    - 64
    - 128
    layers:
    - 1
    - 2
    dropout:
    - 0.1
    - 0.2
    learning_rate:
    - 0.001
    - 0.0005
    batch_size:
    - 32
    - 64
  seed_tuning: 1
  final_seeds:
  - 1
  - 2
  - 3
  epochs: 100
  patience: 10
  loss: mae
  benchmark_max_hours: 12
  benchmark_safety_factor: 1.5
  smoke:
    units: 32
    layers: 1
    dropout: 0.1
    learning_rate: 0.001
    batch_size: 64
    epochs: 2
baseline:
  seasonal_naive_lag: 2016
  moving_average_windows:
  - 3
  - 6
  - 12
  - 24
  arima:
    max_d: 2
    max_p: 5
    max_q: 5
    criterion: aic
evaluation:
  metrics:
  - mae
  - rmse
  - mape
  threshold_celsius: 27.0
  consecutive_alert_steps: 3
  alert_cooldown_minutes: 60
  alpha: 0.05
  dm_final_seed_family: per_seed_per_horizon
  warning_final_protocol: null
runtime:
  require_gpu: true
  memory_growth: true
'''

In [ ]:
# requirements.txt
KODE_PIPELINE['requirements.txt'] = '''numpy==2.0.2
pandas==3.0.6
PyYAML==6.0.3
scipy==1.18.1
statsmodels==0.15.0
tensorflow==2.18.1
'''

In [ ]:
# fungsi_bersama/common.py
KODE_PIPELINE['fungsi_bersama/common.py'] = '''from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml


def load_config(path: str | Path = "config.yaml") -> dict:
    with Path(path).open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def atomic_json(path: str | Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\\n", encoding="utf-8")
    os.replace(temporary, path)


def atomic_csv(path: str | Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def make_run_dir(cfg: dict, requested: str | None = None) -> Path:
    if requested:
        run_dir = Path(requested).expanduser().resolve()
    else:
        stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
        run_dir = (Path(cfg["project"]["run_root"]) / f"candidate_{stamp}").resolve()
    for name in ["config", "dataset", "data", "models", "deployment", "results", "logs", "test_vectors", "state"]:
        (run_dir / name).mkdir(parents=True, exist_ok=True)
    return run_dir


def _timestamps(series: pd.Series, timezone: str) -> pd.Series:
    parsed = pd.to_datetime(series, format="mixed", errors="raise", utc=True)
    return parsed.dt.tz_convert(timezone)


def _interpolate_short(frame: pd.DataFrame, max_slots: int) -> tuple[pd.DataFrame, int]:
    result = frame.copy()
    filled = 0
    for column in result.columns:
        missing = result[column].isna()
        groups = (missing != missing.shift()).cumsum()
        lengths = missing.groupby(groups).transform("size")
        eligible = missing & (lengths <= max_slots)
        interpolated = result[column].interpolate(method="linear", limit=max_slots, limit_area="inside")
        can_fill = eligible & interpolated.notna()
        result.loc[can_fill, column] = interpolated.loc[can_fill]
        filled += int(can_fill.sum())
    return result, filled


def _segments(frame: pd.DataFrame, min_length: int) -> list[pd.DataFrame]:
    valid = frame.notna().all(axis=1)
    groups = (valid != valid.shift()).cumsum()
    return [part for _, part in frame.groupby(groups) if valid.loc[part.index[0]] and len(part) >= min_length]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    hour = result.index.hour + result.index.minute / 60.0
    day = result.index.dayofweek.astype(float)
    result["jam_sin"] = np.sin(2 * np.pi * hour / 24)
    result["jam_cos"] = np.cos(2 * np.pi * hour / 24)
    result["hari_sin"] = np.sin(2 * np.pi * day / 7)
    result["hari_cos"] = np.cos(2 * np.pi * day / 7)
    result["akhir_pekan"] = (result.index.dayofweek >= 5).astype(float)
    return result


def _sequences(segments: list[pd.DataFrame], features: list[str], target: str, lookback: int, horizon: int):
    xs, ys, y_celsius, target_times, issue_times = [], [], [], [], []
    for segment in segments:
        enriched = add_features(segment)
        values = enriched[features].to_numpy(dtype="float32")
        scaled_target = enriched[target].to_numpy(dtype="float32")
        raw_target = enriched[f"{target}_celsius"].to_numpy(dtype="float32")
        for start in range(len(enriched) - lookback - horizon + 1):
            issue = start + lookback - 1
            target_pos = issue + horizon
            xs.append(values[start : start + lookback])
            ys.append(scaled_target[target_pos])
            y_celsius.append(raw_target[target_pos])
            issue_times.append(enriched.index[issue].tz_localize(None).to_datetime64())
            target_times.append(enriched.index[target_pos].tz_localize(None).to_datetime64())
    shape = (0, lookback, len(features))
    return (
        np.stack(xs) if xs else np.empty(shape, dtype="float32"),
        np.asarray(ys, dtype="float32"),
        np.asarray(y_celsius, dtype="float32"),
        np.asarray(issue_times, dtype="datetime64[ns]"),
        np.asarray(target_times, dtype="datetime64[ns]"),
    )


def fit_sensor_scaler(frame: pd.DataFrame, columns: list[str], preprocessing: dict):
    """Fit observed sensor extrema on the configured information boundary."""
    mode = preprocessing.get("scaler_fit", "all")
    if mode == "train_only":
        left, right = map(pd.Timestamp, preprocessing["split"]["train"])
        fit_frame = frame.loc[(frame.index >= left) & (frame.index < right), columns]
    elif mode == "all":
        # Compatibility for immutable historical configs that used global scaling.
        fit_frame = frame[columns]
    else:
        raise ValueError(f"Mode scaler tidak didukung: {mode}")
    minimum, maximum = fit_frame.min(), fit_frame.max()
    if not np.isfinite(minimum.to_numpy()).all() or not np.isfinite(maximum.to_numpy()).all():
        raise ValueError("Data fit scaler kosong atau tidak finite.")
    if ((maximum - minimum) <= 0).any():
        raise ValueError("Rentang Min-Max nol atau negatif.")
    return minimum, maximum


def prepare_dataset(cfg: dict, run_dir: Path, allow_incomplete: bool = False) -> dict:
    data_cfg, pre = cfg["data"], cfg["preprocessing"]
    source = Path(data_cfg["csv_path"]).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {source}")
    if (run_dir / "dataset" / "metadata.json").exists():
        raise FileExistsError("Run sudah dipraproses. Pakai artefak yang ada atau direktori run baru; jangan menimpa dataset run.")
    frame = pd.read_csv(source)
    required = {data_cfg["kolom_waktu"], data_cfg["kolom_target"], data_cfg["kolom_kelembapan"]}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError("Kolom dataset belum lengkap: " + ", ".join(sorted(missing)))
    timezone = cfg["project"]["timezone"]
    frame["timestamp"] = _timestamps(frame[data_cfg["kolom_waktu"]], timezone)
    temperature, humidity = data_cfg["kolom_target"], data_cfg["kolom_kelembapan"]
    frame = frame[["timestamp", temperature, humidity]].copy()
    frame[[temperature, humidity]] = frame[[temperature, humidity]].apply(pd.to_numeric, errors="coerce")
    frame = frame.dropna(subset=["timestamp", temperature, humidity]).sort_values("timestamp")
    raw_first, raw_last = frame["timestamp"].min(), frame["timestamp"].max()
    start, end = pd.Timestamp(data_cfg["mulai"]), pd.Timestamp(data_cfg["akhir_eksklusif"])
    tolerance = pd.Timedelta(minutes=int(data_cfg["toleransi_ujung_menit"]))
    complete = raw_first <= start + tolerance and raw_last >= end - tolerance
    if data_cfg.get("kandidat_harus_lengkap", True) and not allow_incomplete and not complete:
        raise ValueError(
            f"Dataset belum menutup jendela final. Rentang tersedia {raw_first} sampai {raw_last}; "
            f"butuh cakupan {start} sampai mendekati {end}. Gunakan --allow-incomplete hanya untuk smoke test."
        )
    frame = frame[(frame["timestamp"] >= start) & (frame["timestamp"] < end)]
    removed_artifact = 0
    for interval in data_cfg.get("artefak_ranges", []):
        left = pd.Timestamp(interval["mulai_eksklusif"])
        right = pd.Timestamp(interval["selesai_eksklusif"])
        mask = (frame["timestamp"] > left) & (frame["timestamp"] < right)
        removed_artifact += int(mask.sum())
        frame = frame.loc[~mask]
    lo, hi = data_cfg["batas_operasi_suhu"]
    invalid = ~frame[temperature].between(lo, hi)
    removed_invalid = int(invalid.sum())
    frame = frame.loc[~invalid].drop_duplicates(subset="timestamp", keep="first")
    raw_count = len(frame)
    resampled = frame.set_index("timestamp")[[temperature, humidity]].resample(pre["resample"]).mean()
    full_grid = pd.date_range(start=start, end=end, freq=pre["resample"], inclusive="left")
    resampled = resampled.reindex(full_grid)
    resampled.index.name = "timestamp"
    scaler_min, scaler_max = fit_sensor_scaler(resampled, [temperature, humidity], pre)
    span = scaler_max - scaler_min
    if (span <= 0).any():
        raise ValueError("Rentang Min-Max nol atau negatif.")
    scaled = (resampled - scaler_min) / span
    scaled[f"{temperature}_celsius"] = resampled[temperature]
    scaled[f"{humidity}_raw"] = resampled[humidity]
    split_frames, split_segments, rows = {}, {}, []
    max_slots = int(pd.Timedelta(minutes=pre["max_gap_interpolasi_menit"]) / pd.Timedelta(pre["resample"]))
    min_length = pre["lookback"] + max(pre["horizons"])
    for name, bounds in pre["split"].items():
        left, right = pd.Timestamp(bounds[0]), pd.Timestamp(bounds[1])
        part = scaled[(scaled.index >= left) & (scaled.index < right)].copy()
        sensor_columns = [temperature, humidity, f"{temperature}_celsius", f"{humidity}_raw"]
        part[sensor_columns], filled = _interpolate_short(part[sensor_columns], max_slots)
        segments = _segments(part, min_length)
        split_frames[name], split_segments[name] = part, segments
        rows.append({"split": name, "slots": len(part), "filled_cells": filled, "segments": len(segments)})
    prepared = pd.concat([part.assign(split=name) for name, part in split_frames.items()]).reset_index()
    prepared.rename(columns={prepared.columns[0]: "timestamp"}, inplace=True)
    atomic_csv(run_dir / "data" / "prepared_series.csv", prepared)
    features = list(pre["features"])
    sequence_counts = []
    for horizon in pre["horizons"]:
        package = {"features": np.asarray(features), "lookback": pre["lookback"], "horizon": horizon}
        for name in ["train", "val", "test"]:
            x, y, yc, issue, target_time = _sequences(
                split_segments[name], features, temperature, pre["lookback"], horizon
            )
            package.update({f"X_{name}": x, f"y_{name}": y, f"ycel_{name}": yc,
                            f"issue_{name}": issue, f"t_{name}": target_time})
            actual_series = split_frames[name][f"{temperature}_celsius"].copy()
            actual_series.index = actual_series.index.tz_localize(None)
            actual_at_issue = actual_series.reindex(pd.DatetimeIndex(issue)).to_numpy(dtype="float64")
            if not np.isfinite(actual_at_issue).all():
                raise ValueError("Suhu aktual saat issue tidak tersedia untuk sekuens valid.")
            package[f"actual_at_issue_{name}"] = actual_at_issue
            sequence_counts.append({"horizon_minutes": horizon * 5, "split": name, "samples": len(y)})
        np.savez_compressed(run_dir / "data" / f"seq_h{horizon}.npz", **package)
    scaler = {"mode": "fit_on_train" if pre.get("scaler_fit", "all") == "train_only" else "fit_on_all",
              "fit_split": "train" if pre.get("scaler_fit", "all") == "train_only" else "all",
              "clip": False, "columns": [temperature, humidity],
              "min": scaler_min.tolist(), "max": scaler_max.tolist()}
    atomic_json(run_dir / "data" / "scaler.json", scaler)
    atomic_csv(run_dir / "results" / "preprocessing_splits.csv", pd.DataFrame(rows))
    atomic_csv(run_dir / "results" / "sequence_counts.csv", pd.DataFrame(sequence_counts))
    config_snapshot = run_dir / "config" / "config.yaml"
    config_snapshot.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    valid_slots = resampled[[temperature, humidity]].notna().all(axis=1)
    missing_slots = ~valid_slots
    missing_groups = (missing_slots != missing_slots.shift()).cumsum()
    gap_lengths = [len(part) for _, part in resampled.groupby(missing_groups)
                   if missing_slots.loc[part.index[0]]]
    total_slots = len(resampled)
    complete_slots = int(valid_slots.sum())
    completeness = complete_slots / total_slots if total_slots else 0.0
    meta = {
        "status": cfg["project"]["status"], "source": str(source), "source_size_bytes": source.stat().st_size,
        "raw_first": raw_first.isoformat(), "raw_last": raw_last.isoformat(), "final_window_complete": complete,
        "rows_after_filter": raw_count, "removed_artifact_rows": removed_artifact,
        "removed_outside_sensor_range": removed_invalid, "feature_order": features,
        "lookback": pre["lookback"], "horizons": pre["horizons"],
        "total_five_minute_slots": total_slots, "complete_five_minute_slots": complete_slots,
        "missing_five_minute_slots": total_slots - complete_slots,
        "completeness_fraction": completeness, "completeness_target_fraction": 0.95,
        "meets_completeness_target": completeness >= 0.95,
        "long_gap_count": len(gap_lengths), "longest_gap_slots": max(gap_lengths, default=0),
        "meets_no_gap_over_three_hours": max(gap_lengths, default=0) <= 36,
    }
    meta["fingerprint"] = stable_hash({"dataset_metadata": meta, "config": cfg, "schema": 2})
    atomic_json(run_dir / "dataset" / "metadata.json", meta)
    return meta


def load_sequence(run_dir: Path, horizon: int, splits: list[str] | None = None) -> dict[str, np.ndarray]:
    path = run_dir / "data" / f"seq_h{horizon}.npz"
    if not path.is_file():
        raise FileNotFoundError(f"Artefak sekuens belum ada: {path}")
    loaded = np.load(path, allow_pickle=False)
    if splits is None:
        keys = loaded.files
    else:
        suffixes = tuple(f"_{name}" for name in splits)
        keys = [key for key in loaded.files if key in {"features", "lookback", "horizon"} or key.endswith(suffixes)]
    return {key: loaded[key] for key in keys}


def inverse_target(run_dir: Path, values: np.ndarray) -> np.ndarray:
    scaler = json.loads((run_dir / "data" / "scaler.json").read_text())
    lo, hi = float(scaler["min"][0]), float(scaler["max"][0])
    return np.asarray(values) * (hi - lo) + lo


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    error = np.asarray(y_true) - np.asarray(y_pred)
    return {"mae": float(np.mean(np.abs(error))), "rmse": float(np.sqrt(np.mean(error ** 2))),
            "mape": float(np.mean(np.abs(error / y_true)) * 100), "n": int(len(error))}
'''

In [ ]:
# 03_praproses/02_prepare.py
KODE_PIPELINE['03_praproses/02_prepare.py'] = '''from __future__ import annotations

import argparse
from pathlib import Path

import sys
from pathlib import Path
PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / 'fungsi_bersama'))
sys.path.insert(0, str(PIPELINE_ROOT / '04_training'))
import common


def main() -> None:
    parser = argparse.ArgumentParser(description="Praproses PC final; dataset terkunci dan run tidak ditimpa.")
    parser.add_argument("--config", default="config.yaml")
    parser.add_argument("--run-dir")
    parser.add_argument("--allow-incomplete", action="store_true", help="Hanya untuk smoke test.")
    args = parser.parse_args()
    cfg = common.load_config(args.config)
    run_dir = common.make_run_dir(cfg, args.run_dir)
    meta = common.prepare_dataset(cfg, run_dir, allow_incomplete=args.allow_incomplete)
    print(f"RUN_DIR={run_dir}")
    print(f"Dataset: {meta['source']}")
    print(f"Rentang: {meta['raw_first']} sampai {meta['raw_last']}")
    print(f"Jendela final lengkap: {meta['final_window_complete']}")
    print("Jumlah sampel:")
    print((run_dir / "results" / "sequence_counts.csv").read_text())


if __name__ == "__main__":
    main()
'''

In [ ]:
# 04_training/04_train.py
KODE_PIPELINE['04_training/04_train.py'] = '''"""Train and select direct-horizon LSTM/GRU models with immutable run evidence.

Phases: smoke checks fit/save/load; benchmark estimates duration; tune selects
shared hyperparameters using validation; final fits locked choices for each seed.
TensorFlow imports stay inside training functions so provenance can be inspected
without initializing a GPU runtime. Never resume an old run after editing source.
"""

from __future__ import annotations

import argparse
import gc
import itertools
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / "fungsi_bersama"))
sys.path.insert(0, str(PIPELINE_ROOT / "04_training"))
import common
import pc_training_state as state


def build_model(architecture: str, hp: dict, input_shape: tuple[int, int], loss: str):
    """Build one LSTM or GRU model with a scalar direct-horizon output."""
    from tensorflow import keras

    cell = keras.layers.LSTM if architecture == "lstm" else keras.layers.GRU
    model = keras.Sequential(name=f"{architecture}_{hp['config_id']}")
    model.add(keras.Input(shape=input_shape))
    for index in range(hp["layers"]):
        model.add(cell(hp["units"], return_sequences=index < hp["layers"] - 1))
        model.add(keras.layers.Dropout(hp["dropout"]))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(hp["learning_rate"]), loss=loss)
    return model


def grid(cfg: dict) -> list[dict]:
    """Enumerate configured candidates in stable order without choosing a winner."""
    values = cfg["model"]["grid"]
    result = []
    for units, layers, dropout, lr, batch in itertools.product(
        values["units"],
        values["layers"],
        values["dropout"],
        values["learning_rate"],
        values["batch_size"],
    ):
        hp = {
            "units": units,
            "layers": layers,
            "dropout": float(dropout),
            "learning_rate": float(lr),
            "batch_size": batch,
        }
        hp["config_id"] = f"u{units}_l{layers}_d{dropout}_lr{lr}_b{batch}".replace(
            ".", "p"
        )
        result.append(hp)
    return result


def environment_snapshot(run_dir: Path, cfg: dict) -> dict:
    """Verify GPU availability and record the exact environment used for fitting."""
    import importlib.metadata as metadata
    import tensorflow as tf

    packages = [
        "tensorflow",
        "tensorflow-metal",
        "keras",
        "numpy",
        "pandas",
        "scipy",
        "statsmodels",
        "scikit-learn",
        "PyYAML",
        "jupyterlab",
    ]
    versions = {}
    for package in packages:
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = None
    versions.update(
        {
            d.metadata["Name"]: d.version
            for d in metadata.distributions()
            if d.metadata["Name"]
        }
    )
    devices = tf.config.list_physical_devices("GPU")
    if cfg.get("runtime", {}).get("require_gpu", False) and not devices:
        raise RuntimeError(
            "GPU diwajibkan untuk run PC ini; TensorFlow tidak mendeteksi GPU."
        )
    if cfg.get("runtime", {}).get("memory_growth", True):
        for device in devices:
            tf.config.experimental.set_memory_growth(device, True)
    gpus = [device.name for device in devices]
    with tf.device("/GPU:0" if gpus else "/CPU:0"):
        probe = tf.matmul(tf.ones((16, 16)), tf.ones((16, 16)))
    snapshot = {
        "python": sys.version,
        "executable": sys.executable,
        "platform": platform.platform(),
        "packages": versions,
        "physical_gpus": gpus,
        "probe_device": probe.device,
        "same_process_as_model_fit": True,
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    import subprocess

    try:
        gpu_runtime = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=uuid,name,driver_version,memory.total",
                "--format=csv,noheader",
            ],
            text=True,
            timeout=20,
        ).strip()
    except (OSError, subprocess.SubprocessError):
        gpu_runtime = "unavailable"
    snapshot["gpu_runtime"] = gpu_runtime
    snapshot["hostname"] = platform.node()
    snapshot["machine"] = platform.machine()
    snapshot["tensorflow_build"] = tf.sysconfig.get_build_info()
    snapshot["gpu_details"] = [
        tf.config.experimental.get_device_details(d)
        for d in tf.config.list_physical_devices("GPU")
    ]
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    common.atomic_json(run_dir / "logs" / "environments" / f"{stamp}.json", snapshot)
    return snapshot


def _row_key(
    fingerprint: str, architecture: str, horizon: int, hp: dict, seed: int
) -> str:
    """Hash the run, model, horizon, hyperparameters, and seed into one fit key."""
    return common.stable_hash(
        {
            "fingerprint": fingerprint,
            "architecture": architecture,
            "horizon": horizon,
            "hp": hp,
            "seed": seed,
        }
    )


def load_progress(path: Path) -> pd.DataFrame:
    """Read recorded trials, returning an empty table for a fresh phase."""
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_csv(path)


def train_once(
    cfg: dict,
    sequence: dict,
    architecture: str,
    hp: dict,
    seed: int,
    epochs: int | None = None,
    subset: int | None = None,
    attempt_dir: Path | None = None,
):
    """Fit only training data, select the best validation checkpoint, and return fit evidence."""
    import tensorflow as tf
    from tensorflow import keras

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    x_train, y_train = sequence["X_train"], sequence["y_train"]
    x_val, y_val = sequence["X_val"], sequence["y_val"]
    if subset:
        x_train, y_train = x_train[:subset], y_train[:subset]
        x_val, y_val = x_val[: max(64, subset // 4)], y_val[: max(64, subset // 4)]
    if not len(x_train) or not len(x_val):
        raise ValueError("Training dan validasi harus memiliki sampel.")
    model = build_model(architecture, hp, x_train.shape[1:], cfg["model"]["loss"])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=cfg["model"]["patience"],
            restore_best_weights=True,
            verbose=0,
        )
    ]
    if attempt_dir:
        attempt_dir.mkdir(parents=True, exist_ok=True)
        callbacks += [
            keras.callbacks.CSVLogger(str(attempt_dir / "history.csv")),
            keras.callbacks.ModelCheckpoint(
                str(attempt_dir / "best.keras"),
                monitor="val_loss",
                save_best_only=True,
                verbose=0,
            ),
            keras.callbacks.TerminateOnNaN(),
        ]
    started = time.perf_counter()
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs or cfg["model"]["epochs"],
        batch_size=int(hp["batch_size"]),
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )
    duration = time.perf_counter() - started
    losses = np.asarray(history.history["val_loss"])
    if not np.isfinite(losses).all():
        raise RuntimeError("Loss non-finite; trial gagal dan tidak boleh dipilih.")
    # Reload the persisted best checkpoint, including optimizer state, for final artefacts.
    if attempt_dir:
        model = keras.models.load_model(attempt_dir / "best.keras")
    prediction = model.predict(x_val, verbose=0).reshape(-1)
    if not np.isfinite(prediction).all():
        raise RuntimeError("Prediksi validasi non-finite.")
    predicted_celsius = common.inverse_target(Path(cfg["_run_dir"]), prediction)
    val_mae = common.metrics(
        sequence["ycel_val"][: len(prediction)], predicted_celsius
    )["mae"]
    details = {
        "val_mae_celsius": val_mae,
        "duration_seconds": duration,
        "epochs_ran": len(history.history["loss"]),
        "best_epoch": int(np.argmin(losses) + 1),
        "trainable_params": int(model.count_params()),
        "variable_devices": sorted(
            {
                str(getattr(getattr(v, "value", v), "device", "unknown"))
                for v in model.trainable_variables
            }
        ),
        "same_process_as_model_fit": True,
        "op_determinism_enabled": True,
        "seed": seed,
    }
    if attempt_dir:
        common.atomic_json(attempt_dir / "fit_summary.json", details)
    return model, details


def append_result(path: Path, row: dict) -> pd.DataFrame:
    """Persist the latest status for each trial without discarding other trials."""
    frame = load_progress(path)
    frame = pd.concat([frame, pd.DataFrame([row])], ignore_index=True)
    frame = frame.drop_duplicates(subset=["trial_id"], keep="last")
    common.atomic_csv(path, frame)
    return frame


def run_trial(
    cfg: dict,
    run_dir: Path,
    fingerprint: str,
    architecture: str,
    horizon: int,
    hp: dict,
    progress_path: Path,
    force: bool = False,
    seed: int | None = None,
    phase: str = "tuning",
):
    """Reuse a verified completed fit or restart an interrupted fit from its original seed."""
    seed = int(cfg["model"]["seed_tuning"] if seed is None else seed)
    trial_id = common.stable_hash(
        {"phase": phase, "fit": _row_key(fingerprint, architecture, horizon, hp, seed)}
    )
    progress = load_progress(progress_path)
    if not force and not progress.empty:
        done = progress[
            (progress["trial_id"] == trial_id) & (progress["status"] == "success")
        ]
        if not done.empty:
            row = done.iloc[-1].to_dict()
            state.verify_artifacts(run_dir, row)
            return None, row
    # A stopped fit is rerun from the original seed. RNG/dropout and EarlyStopping
    # state are not reconstructed from a best-only checkpoint; never pretend an
    # epoch continuation is scientifically identical. Keep every attempt.
    attempt_dir = state.new_attempt(run_dir, phase, trial_id)
    sequence = common.load_sequence(run_dir, horizon, splits=["train", "val"])
    row = {
        "trial_id": trial_id,
        "fingerprint": fingerprint,
        "status": "running",
        "phase": phase,
        "architecture": architecture,
        "horizon_slots": horizon,
        "horizon_minutes": horizon * 5,
        "seed": seed,
        **hp,
        "started_at_utc": state.utc_now(),
        "attempt_dir": str(attempt_dir.relative_to(run_dir)),
        "error": "",
    }
    common.atomic_json(attempt_dir / "status.json", row)
    append_result(progress_path, row)
    try:
        model, details = train_once(
            cfg, sequence, architecture, hp, seed, attempt_dir=attempt_dir
        )
        model_path = attempt_dir / "model.keras"
        model.save(model_path)
        row.update(details)
        row.update(
            {
                "status": "success",
                "finished_at_utc": state.utc_now(),
                "model_path": str(model_path.relative_to(run_dir)),
                "model_sha256": common.sha256_file(model_path),
                "history_path": str((attempt_dir / "history.csv").relative_to(run_dir)),
                "history_sha256": common.sha256_file(attempt_dir / "history.csv"),
                "checkpoint_path": str(
                    (attempt_dir / "best.keras").relative_to(run_dir)
                ),
                "checkpoint_sha256": common.sha256_file(attempt_dir / "best.keras"),
            }
        )
        common.atomic_json(attempt_dir / "status.json", row)
        append_result(progress_path, row)
        return model, row
    except BaseException as exc:
        row.update(
            {
                "status": (
                    "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
                ),
                "finished_at_utc": state.utc_now(),
                "error": repr(exc),
            }
        )
        common.atomic_json(attempt_dir / "status.json", row)
        append_result(progress_path, row)
        raise


def shared_selection(progress: pd.DataFrame, cfg: dict, fingerprint: str) -> dict:
    """Select shared LSTM/GRU hyperparameters from tuning-validation MAE only."""
    if progress.empty:
        return {}
    successful = progress[
        (progress["status"] == "success")
        & (progress["fingerprint"] == fingerprint)
        & (progress["seed"] == int(cfg["model"]["seed_tuning"]))
        & (progress["phase"] == "tuning")
    ].copy()
    selections = {}
    hp_columns = [
        "units",
        "layers",
        "dropout",
        "learning_rate",
        "batch_size",
        "config_id",
    ]
    for horizon in cfg["preprocessing"]["horizons"]:
        subset = successful[successful["horizon_slots"] == horizon]
        pivot = subset.pivot_table(
            index="config_id",
            columns="architecture",
            values="val_mae_celsius",
            aggfunc="last",
        )
        if not {"lstm", "gru"}.issubset(pivot.columns):
            continue
        pivot = pivot.dropna(subset=["lstm", "gru"])
        if pivot.empty:
            continue
        pivot["mean"] = (pivot["lstm"] + pivot["gru"]) / 2
        pivot["worst"] = pivot[["lstm", "gru"]].max(axis=1)
        best_id = (
            pivot.reset_index()
            .sort_values(["mean", "worst", "config_id"])
            .iloc[0]["config_id"]
        )
        source = subset[subset["config_id"] == best_id].iloc[0]
        hp = {
            column: (
                source[column].item()
                if hasattr(source[column], "item")
                else source[column]
            )
            for column in hp_columns
        }
        winner = (
            "lstm" if pivot.loc[best_id, "lstm"] <= pivot.loc[best_id, "gru"] else "gru"
        )
        selections[str(horizon)] = {
            "horizon_minutes": horizon * 5,
            "shared_hyperparameters": hp,
            "lstm_val_mae": float(pivot.loc[best_id, "lstm"]),
            "gru_val_mae": float(pivot.loc[best_id, "gru"]),
            "pair_mean_val_mae": float(pivot.loc[best_id, "mean"]),
            "deployment_winner": winner,
            "seed": int(cfg["model"]["seed_tuning"]),
        }
    return selections


def final_training(cfg: dict, run_dir: Path, fingerprint: str) -> None:
    """Fit locked selections for all final seeds and export auditable prediction artifacts."""
    import tensorflow as tf

    seeds = [int(seed) for seed in cfg["model"]["final_seeds"]]
    if seeds != [1, 2, 3]:
        raise ValueError("Protokol final membutuhkan final_seeds tepat [1, 2, 3].")
    path = run_dir / "results" / "selected_shared_configs.json"
    selection = json.loads(path.read_text(encoding="utf-8"))
    if selection.get("run_fingerprint") != fingerprint:
        raise RuntimeError(
            "Selection berasal dari run/runtime berbeda; jangan reuse tuning Metal."
        )
    selections = selection["selections"]
    if set(selections) != {str(h) for h in cfg["preprocessing"]["horizons"]}:
        raise RuntimeError("Selection belum lengkap untuk seluruh horizon.")
    lock_path = run_dir / "state" / "locked_final_selection.json"
    state.lock_json(lock_path, selection)
    progress_path = run_dir / "results" / "final_training_progress.csv"
    manifest = []
    predictions = []
    for horizon in cfg["preprocessing"]["horizons"]:
        hp = selections[str(horizon)]["shared_hyperparameters"]
        for seed in seeds:
            payload = {}
            for architecture in cfg["model"]["architectures"]:
                model, row = run_trial(
                    cfg,
                    run_dir,
                    fingerprint,
                    architecture,
                    horizon,
                    hp,
                    progress_path,
                    seed=seed,
                    phase="final",
                )
                if model is None:
                    model = tf.keras.models.load_model(run_dir / row["model_path"])
                # Test data is loaded only after the fit and config choice are complete.
                sequence = common.load_sequence(run_dir, horizon)
                payload.update(
                    {
                        "y_true": sequence["ycel_test"],
                        "t": sequence["t_test"],
                        "issue": sequence["issue_test"],
                        "y_true_val": sequence["ycel_val"],
                        "t_val": sequence["t_val"],
                        "issue_val": sequence["issue_val"],
                        "run_fingerprint": np.asarray(fingerprint),
                        "seed": np.asarray(seed),
                    }
                )
                for split in ["val", "test"]:
                    payload[f"actual_at_issue_{split}"] = sequence[
                        f"actual_at_issue_{split}"
                    ]
                    pred = model.predict(sequence[f"X_{split}"], verbose=0).reshape(-1)
                    if not np.isfinite(pred).all():
                        raise RuntimeError("Prediksi non-finite; evaluasi dihentikan.")
                    payload[f"{architecture}_{split}"] = common.inverse_target(
                        run_dir, pred
                    )
                canonical = (
                    run_dir
                    / "models"
                    / f"{architecture}_h{horizon * 5}_seed{seed}.keras"
                )
                state.copy_verified(
                    run_dir / row["model_path"], canonical, row["model_sha256"]
                )
                manifest.append(
                    {
                        "architecture": architecture,
                        "horizon_slots": horizon,
                        "horizon_minutes": horizon * 5,
                        "seed": seed,
                        "model_path": str(canonical.relative_to(run_dir)),
                        "model_sha256": row["model_sha256"],
                        "trial_id": row["trial_id"],
                        "val_mae_celsius": row["val_mae_celsius"],
                        "duration_seconds": row["duration_seconds"],
                        "history_path": row["history_path"],
                        "attempt_dir": row["attempt_dir"],
                    }
                )
                del model, sequence
                tf.keras.backend.clear_session()
                gc.collect()
            prediction_path = (
                run_dir / "results" / f"neural_predictions_h{horizon}_seed{seed}.npz"
            )
            state.atomic_npz(prediction_path, payload)
            predictions.append(
                {
                    "path": str(prediction_path.relative_to(run_dir)),
                    "sha256": common.sha256_file(prediction_path),
                    "horizon_slots": horizon,
                    "seed": seed,
                }
            )
    common.atomic_json(
        run_dir / "results" / "final_models_manifest.json",
        {
            "run_fingerprint": fingerprint,
            "seeds": seeds,
            "models": manifest,
            "predictions": predictions,
            "deployment_selection": "PENDING_PREDECLARED_SEED_OR_ENSEMBLE_POLICY",
            "selection_basis": "shared_configuration_from_tuning_validation_only",
        },
    )


def smoke(cfg: dict, run_dir: Path) -> None:
    """Check a small fit and save/load cycle before running the full experiment."""
    import tensorflow as tf

    hp = dict(cfg["model"]["smoke"])
    epochs = int(hp.pop("epochs"))
    hp["config_id"] = "smoke_u32_l1_d0p1"
    sequence = common.load_sequence(
        run_dir, max(cfg["preprocessing"]["horizons"]), splits=["train", "val"]
    )
    results = []
    for architecture in cfg["model"]["architectures"]:
        attempt_dir = state.new_attempt(run_dir, "smoke", architecture)
        model, row = train_once(
            cfg,
            sequence,
            architecture,
            hp,
            cfg["model"]["seed_tuning"],
            epochs=epochs,
            subset=256,
            attempt_dir=attempt_dir,
        )
        path = run_dir / "state" / f"smoke_{architecture}.keras"
        model.save(path)
        tf.keras.models.load_model(path)
        results.append({"architecture": architecture, **row, "model_save_load": True})
        tf.keras.backend.clear_session()
    common.atomic_json(run_dir / "logs" / "smoke_test.json", results)


def benchmark(cfg: dict, run_dir: Path, fingerprint: str, progress_path: Path) -> dict:
    """Estimate total tuning duration from representative, reusable tuning trials."""
    candidates = grid(cfg)
    ids = {hp["config_id"]: hp for hp in candidates}
    selected = list(
        {
            hp["config_id"]: hp
            for hp in [candidates[len(candidates) // 2], candidates[-1]]
        }.values()
    )
    durations = []
    for horizon in cfg["preprocessing"]["horizons"]:
        for hp in selected:
            for architecture in cfg["model"]["architectures"]:
                model, row = run_trial(
                    cfg, run_dir, fingerprint, architecture, horizon, hp, progress_path
                )
                durations.append(float(row["duration_seconds"]))
                if model is not None:
                    del model
                gc.collect()
    estimate = float(
        np.mean(durations)
        * len(candidates)
        * len(cfg["model"]["architectures"])
        * len(cfg["preprocessing"]["horizons"])
        * cfg["model"]["benchmark_safety_factor"]
        / 3600
    )
    result = {
        "sample_trials": len(durations),
        "mean_seconds": float(np.mean(durations)),
        "safety_factor": cfg["model"]["benchmark_safety_factor"],
        "estimated_total_hours": estimate,
        "limit_hours": cfg["model"]["benchmark_max_hours"],
        "may_continue": estimate <= cfg["model"]["benchmark_max_hours"],
    }
    common.atomic_json(run_dir / "logs" / "benchmark.json", result)
    return result


def tune(cfg: dict, run_dir: Path, fingerprint: str, progress_path: Path) -> None:
    """Complete the configured grid and lock one shared configuration per horizon."""
    for horizon in cfg["preprocessing"]["horizons"]:
        for hp in grid(cfg):
            for architecture in cfg["model"]["architectures"]:
                model, _ = run_trial(
                    cfg, run_dir, fingerprint, architecture, horizon, hp, progress_path
                )
                if model is not None:
                    del model
                gc.collect()
    progress = load_progress(progress_path)
    selections = shared_selection(progress, cfg, fingerprint)
    if len(selections) != len(cfg["preprocessing"]["horizons"]):
        raise RuntimeError(
            "Tuning belum lengkap; konfigurasi bersama belum tersedia untuk semua horizon."
        )
    state.lock_json(
        run_dir / "results" / "selected_shared_configs.json",
        {
            "run_fingerprint": fingerprint,
            "selection_basis": "tuning_validation_pair_mean_mae",
            "selections": selections,
        },
    )


def main() -> None:
    """Validate prepared inputs, bind immutable run identity, and dispatch one phase."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="config.yaml")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument(
        "--mode", choices=["smoke", "benchmark", "tune", "final"], required=True
    )
    args = parser.parse_args()
    cfg = common.load_config(args.config)
    run_dir = Path(args.run_dir).resolve()
    cfg["_run_dir"] = str(run_dir)
    prepared_cfg = common.load_config(run_dir / "config" / "config.yaml")
    stable_cfg = {k: v for k, v in cfg.items() if not k.startswith("_")}
    prepared_cfg = {k: v for k, v in prepared_cfg.items() if not k.startswith("_")}
    if common.stable_hash(stable_cfg) != common.stable_hash(prepared_cfg):
        raise RuntimeError(
            "Config saat ini berbeda dari config praproses; buat run baru."
        )
    metadata = json.loads((run_dir / "dataset" / "metadata.json").read_text())
    expected = cfg["data"].get("expected_sha256")
    if expected and metadata["sha256"] != expected:
        raise RuntimeError("Dataset run tidak cocok checksum final.")
    workspace = os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    if workspace not in {":4096:8", ":16:8"}:
        raise RuntimeError(
            "CUBLAS_WORKSPACE_CONFIG harus deterministik sebelum TensorFlow import."
        )
    if not metadata.get("final_window_complete", False) and args.mode != "smoke":
        raise RuntimeError(
            "Data incomplete tidak dapat dipakai untuk training final/tuning."
        )
    fingerprint = state.bind_run_identity(
        run_dir, cfg, metadata, environment_snapshot(run_dir, cfg)
    )
    progress_path = run_dir / "results" / "tuning_progress.csv"
    if args.mode == "smoke":
        smoke(cfg, run_dir)
    elif args.mode == "benchmark":
        print(json.dumps(benchmark(cfg, run_dir, fingerprint, progress_path), indent=2))
    elif args.mode == "tune":
        tune(cfg, run_dir, fingerprint, progress_path)
    else:
        final_training(cfg, run_dir, fingerprint)


if __name__ == "__main__":
    with state.process_lock():
        main()
'''

In [ ]:
# 04_training/pc_training_state.py
KODE_PIPELINE['04_training/pc_training_state.py'] = '''"""Training provenance and safe resume; no TensorFlow import needed to audit state."""

from __future__ import annotations

from contextlib import contextmanager
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import sys
import uuid

PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / "fungsi_bersama"))
sys.path.insert(0, str(PIPELINE_ROOT / "04_training"))
import common


def utc_now() -> str:
    """Return a timezone-aware UTC timestamp for training provenance."""
    return datetime.now(timezone.utc).isoformat()


def lock_json(path: Path, value: dict) -> None:
    """An existing identity/selection is immutable, never silently replaced."""
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if common.stable_hash(old) != common.stable_hash(value):
            raise RuntimeError(
                f"Identitas/selection berubah: {path}. Gunakan run baru."
            )
    else:
        common.atomic_json(path, value)


def bind_run_identity(
    run_dir: Path, cfg: dict, metadata: dict, environment: dict
) -> str:
    """Lock config, prepared files, source hashes, and runtime into one run fingerprint."""
    source_dir = PIPELINE_ROOT
    sources = {
        str(p.relative_to(source_dir)): common.sha256_file(p)
        for folder in [
            "03_praproses",
            "04_training",
            "05_baseline",
            "06_evaluasi",
            "fungsi_bersama",
        ]
        for p in sorted((source_dir / folder).glob("*.py"))
    }
    prepared = {
        str(p.relative_to(run_dir)): common.sha256_file(p)
        for p in sorted((run_dir / "data").glob("*"))
        if p.is_file()
    }
    stable_cfg = {k: v for k, v in cfg.items() if not k.startswith("_")}
    stable_env = {
        k: v
        for k, v in environment.items()
        if k not in {"captured_at_utc", "same_process_as_model_fit"}
    }
    payload = {
        "schema": 2,
        "dataset_source": metadata.get("source"),
        "dataset_size_bytes": metadata.get("source_size_bytes"),
        "preprocessing_fingerprint": metadata["fingerprint"],
        "prepared_files": prepared,
        "config": stable_cfg,
        "sources": sources,
        "environment": stable_env,
        "resume_policy": "completed_fit_hash_verified; interrupted_fit_restart_seed",
    }
    fingerprint = common.stable_hash(payload)
    identity = {"fingerprint": fingerprint, **payload}
    lock_json(run_dir / "state" / "training_identity.json", identity)
    snapshot = run_dir / "logs" / "training_source_snapshot"
    snapshot.mkdir(parents=True, exist_ok=True)
    for name, digest in sources.items():
        copy_verified(source_dir / name, snapshot / name, digest)
    return fingerprint


def new_attempt(run_dir: Path, phase: str, trial_id: str) -> Path:
    """Create a unique attempt directory so interrupted fits retain their evidence."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    directory = (
        run_dir / "logs" / "fits" / phase / trial_id / f"{stamp}_{uuid.uuid4().hex[:8]}"
    )
    directory.mkdir(parents=True, exist_ok=False)
    return directory


def verify_artifacts(run_dir: Path, row: dict) -> None:
    """Reject missing, escaped, or changed artifacts before reusing a completed fit."""
    for prefix in ["model", "history", "checkpoint"]:
        relative = Path(str(row.get(f"{prefix}_path", "")))
        if not str(relative) or relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Path artefak progress tidak valid.")
        path = (run_dir / relative).resolve()
        if not path.is_relative_to(run_dir.resolve()) or not path.is_file():
            raise RuntimeError(f"Artefak completed fit hilang: {path}")
        if common.sha256_file(path) != row.get(f"{prefix}_sha256"):
            raise RuntimeError(f"Checksum artefak berubah: {path}; gunakan run baru.")


def copy_verified(source: Path, target: Path, digest: str) -> None:
    """Copy an artifact atomically without replacing a different existing artifact."""
    if common.sha256_file(source) != digest:
        raise RuntimeError(f"Sumber berubah: {source}")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if common.sha256_file(target) != digest:
            raise RuntimeError(f"Artefak tujuan berubah: {target}; tidak ditimpa.")
        return
    temporary = target.with_name(target.name + ".tmp")
    shutil.copyfile(source, temporary)
    if common.sha256_file(temporary) != digest:
        temporary.unlink()
        raise RuntimeError("Copy artefak gagal checksum.")
    os.replace(temporary, target)


def atomic_npz(path: Path, payload: dict) -> None:
    """Write compressed prediction arrays through an atomic file replacement."""
    import numpy as np

    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **payload)
    os.replace(temporary, path)


@contextmanager
def process_lock():
    """OS releases flock after a crash; only one writer per run (Linux/WSL)."""
    import fcntl

    parser_run = next(
        (
            sys.argv[i + 1]
            for i, value in enumerate(sys.argv[:-1])
            if value == "--run-dir"
        ),
        None,
    )
    if parser_run is None:
        parser_run = next(
            (
                value.split("=", 1)[1]
                for value in sys.argv
                if value.startswith("--run-dir=")
            ),
            None,
        )
    if parser_run is None:
        yield  # argparse will explain the missing required argument.
        return
    run_dir = Path(parser_run).resolve()
    (run_dir / "state").mkdir(parents=True, exist_ok=True)
    with (run_dir / "state" / "training.lock").open("a+") as handle:
        try:
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Proses training lain sedang menulis run ini.") from exc
        try:
            yield
        finally:
            fcntl.flock(handle.fileno(), fcntl.LOCK_UN)
'''

In [ ]:
# 04_training/job_control.py
KODE_PIPELINE['04_training/job_control.py'] = '''"""Detached WSL training jobs with explicit phase/status; no automatic next phase."""

from __future__ import annotations
import argparse
import json
import os
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / "fungsi_bersama"))
sys.path.insert(0, str(PIPELINE_ROOT / "04_training"))
import common

PHASES = {"smoke", "benchmark", "tune", "final"}


def now():
    """Return a timezone-aware UTC timestamp for job status records."""
    return datetime.now(timezone.utc).isoformat()


def process_token(pid):
    """Identify a live Linux process by boot ID and start time, avoiding PID reuse."""
    try:
        stat = Path(f"/proc/{pid}/stat").read_text().split(") ", 1)[1].split()
        if stat[0] == "Z":
            return None
        return (
            Path("/proc/sys/kernel/random/boot_id").read_text().strip() + ":" + stat[19]
        )
    except (FileNotFoundError, ProcessLookupError):
        return None


def start(phase, run_dir, config):
    """Launch one detached phase and preserve its console output in a new job directory."""
    if phase not in PHASES:
        raise ValueError("Unknown training phase")
    jobs = run_dir / "logs/jobs"
    jobs.mkdir(parents=True, exist_ok=True)
    for path in jobs.glob("*/status.json"):
        record = json.loads(path.read_text())
        if (
            record.get("status") == "running"
            and record.get("process_token")
            and process_token(record["pid"]) == record["process_token"]
        ):
            raise RuntimeError("A training job is already running for this run.")
    job = jobs / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        + "_"
        + phase
        + "_"
        + uuid.uuid4().hex[:8]
    )
    job.mkdir()
    common.atomic_json(
        job / "status.json",
        {"status": "starting", "phase": phase, "requested_at": now()},
    )
    with (job / "console.log").open("wb") as log:
        process = subprocess.Popen(
            [
                sys.executable,
                str(Path(__file__).resolve()),
                "worker",
                "--phase",
                phase,
                "--run-dir",
                str(run_dir),
                "--config",
                str(config),
                "--job-dir",
                str(job),
            ],
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    result = {
        "phase": phase,
        "worker_pid": process.pid,
        "job_dir": str(job),
        "console_log": str(job / "console.log"),
    }
    print(json.dumps(result, indent=2))
    return result


def worker(phase, run_dir, config, job):
    """Run one training phase and always persist its terminal status."""
    record = {
        "status": "running",
        "phase": phase,
        "pid": os.getpid(),
        "process_token": process_token(os.getpid()),
        "started_at": now(),
    }
    common.atomic_json(job / "status.json", record)
    environment = os.environ.copy()
    environment.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    environment.setdefault("PYTHONHASHSEED", "1")
    try:
        command = [
            sys.executable,
            "-u",
            str(Path(__file__).with_name("04_train.py")),
            "--config",
            str(config),
            "--run-dir",
            str(run_dir),
            "--mode",
            phase,
        ]
        result = subprocess.run(command, cwd=PIPELINE_ROOT, env=environment)
        record.update(
            status="completed" if result.returncode == 0 else "failed",
            exit_code=result.returncode,
            finished_at=now(),
        )
    except BaseException as exc:
        record.update(status="failed", error=repr(exc), finished_at=now())
        raise
    finally:
        common.atomic_json(job / "status.json", record)


def main():
    """Dispatch a phase launcher or its internal worker from command-line arguments."""
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=["start", "worker"])
    parser.add_argument("--phase", choices=sorted(PHASES), required=True)
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--config", default="config.yaml")
    parser.add_argument("--job-dir")
    args = parser.parse_args()
    run = Path(args.run_dir).resolve()
    config = Path(args.config).resolve()
    if args.action == "start":
        start(args.phase, run, config)
    else:
        worker(args.phase, run, config, Path(args.job_dir).resolve())


if __name__ == "__main__":
    main()
'''

In [ ]:
# 04_training/42_run_pipeline.py
KODE_PIPELINE['04_training/42_run_pipeline.py'] = '''"""Run the Mac-authored 42-day pipeline as a detached WSL process."""

from __future__ import annotations

import argparse
from datetime import datetime, timezone
import fcntl
import json
import os
from pathlib import Path
import subprocess
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "fungsi_bersama"))
import common


def validate_config(cfg):
    """Require the locked 42-day split, final seeds, and GPU policy before launch."""
    import pandas as pd

    splits = cfg["preprocessing"]["split"]
    expected = {
        "train": ("2026-08-20", "2026-09-17", 28),
        "val": ("2026-09-17", "2026-09-24", 7),
        "test": ("2026-09-24", "2026-10-01", 7),
    }
    for name, (left, right, days) in expected.items():
        actual = [pd.Timestamp(x) for x in splits[name]]
        required = [pd.Timestamp(x, tz="Asia/Jakarta") for x in (left, right)]
        if actual != required or (actual[1] - actual[0]).days != days:
            raise ValueError("Config is not the locked 42-day split.")
    if pd.Timestamp(cfg["data"]["mulai"]) != pd.Timestamp(splits["train"][0]):
        raise ValueError("Dataset start differs from train start.")
    if pd.Timestamp(cfg["data"]["akhir_eksklusif"]) != pd.Timestamp(splits["test"][1]):
        raise ValueError("Dataset end differs from test end.")
    if cfg["model"]["final_seeds"] != [1, 2, 3]:
        raise ValueError("Final seeds must be 1, 2, 3.")
    if not cfg.get("runtime", {}).get("require_gpu"):
        raise ValueError("GPU must be required for this PC experiment.")


def write_status(run, status, **fields):
    """Persist and print the current controller status with UTC provenance."""
    result = {
        "status": status,
        "pid": os.getpid(),
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_dir": str(run),
        "code_authority": "Mac",
        **fields,
    }
    common.atomic_json(run / "logs" / "pipeline_status.json", result)
    print(json.dumps(result), flush=True)


def worker(run, config):
    """Run preparation and training phases under one lock, then verify the final manifest."""
    os.chdir(ROOT)
    cfg = common.load_config(config)
    validate_config(cfg)
    run.mkdir(parents=True, exist_ok=True)
    with (run / "pipeline.lock").open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        status_path = run / "logs" / "pipeline_status.json"
        if (
            status_path.exists()
            and json.loads(status_path.read_text()).get("status")
            == "FINAL_TRAINING_COMPLETED"
        ):
            raise RuntimeError("This run is already complete; do not launch it again.")
        environment = os.environ.copy()
        environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        environment["PYTHONHASHSEED"] = "1"
        try:
            if not (run / "dataset" / "metadata.json").exists():
                write_status(run, "RUNNING_PREPROCESSING")
                subprocess.run(
                    [
                        sys.executable,
                        "-u",
                        "03_praproses/02_prepare.py",
                        "--config",
                        str(config),
                        "--run-dir",
                        str(run),
                    ],
                    env=environment,
                    check=True,
                )
            for phase in ("smoke", "benchmark", "tune", "final"):
                write_status(run, "RUNNING_" + phase.upper(), phase=phase)
                subprocess.run(
                    [
                        sys.executable,
                        "-u",
                        "04_training/04_train.py",
                        "--config",
                        str(config),
                        "--run-dir",
                        str(run),
                        "--mode",
                        phase,
                    ],
                    env=environment,
                    check=True,
                )
                if phase == "benchmark":
                    estimate = json.loads((run / "logs" / "benchmark.json").read_text())
                    if not estimate["may_continue"]:
                        write_status(run, "WAITING_DURATION_REVIEW", benchmark=estimate)
                        return
            manifest = json.loads(
                (run / "results" / "final_models_manifest.json").read_text()
            )
            if len(manifest["models"]) != 18:
                raise RuntimeError("Final manifest does not contain 18 models.")
            for item in manifest["models"]:
                path = run / item["model_path"]
                if common.sha256_file(path) != item["model_sha256"]:
                    raise RuntimeError(
                        "Final model integrity check failed: " + str(path)
                    )
            write_status(
                run,
                "FINAL_TRAINING_COMPLETED",
                models=18,
                evaluation="NOT_STARTED",
                deployment="NOT_CHANGED",
            )
        except BaseException as exc:
            write_status(run, "FAILED", error=repr(exc))
            raise


def main():
    """Start a fresh detached run, execute its worker, or read its saved status."""
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=("start", "worker", "status"))
    parser.add_argument("--config", default="config.yaml")
    parser.add_argument("--run-dir", required=True)
    args = parser.parse_args()
    os.chdir(ROOT)
    run = Path(args.run_dir).resolve()
    config = Path(args.config).resolve()
    if args.action == "status":
        print((run / "logs" / "pipeline_status.json").read_text())
    elif args.action == "worker":
        worker(run, config)
    else:
        validate_config(common.load_config(config))
        # A fresh run is reserved atomically, so repeated clicks cannot start it twice.
        run.mkdir(parents=True, exist_ok=False)
        (run / "logs").mkdir()
        write_status(run, "STARTING")
        with (run / "logs" / "pipeline_console.log").open("wb") as log:
            process = subprocess.Popen(
                [
                    sys.executable,
                    "-u",
                    str(Path(__file__).resolve()),
                    "worker",
                    "--config",
                    str(config),
                    "--run-dir",
                    str(run),
                ],
                cwd=ROOT,
                stdin=subprocess.DEVNULL,
                stdout=log,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )
        print(
            json.dumps({"controller_pid": process.pid, "run_dir": str(run)}, indent=2)
        )


if __name__ == "__main__":
    main()
'''

In [ ]:
# 05_baseline/03_baselines.py
KODE_PIPELINE['05_baseline/03_baselines.py'] = '''from __future__ import annotations

import argparse
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import sys
from pathlib import Path
PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / 'fungsi_bersama'))
sys.path.insert(0, str(PIPELINE_ROOT / '04_training'))
import common


def _series(run_dir: Path, target: str) -> pd.Series:
    frame = pd.read_csv(run_dir / "data" / "prepared_series.csv")
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)
    return frame.set_index("timestamp")[f"{target}_celsius"].sort_index()


def _sequence_index(times: np.ndarray, timezone: str) -> pd.DatetimeIndex:
    index = pd.DatetimeIndex(pd.to_datetime(times))
    if index.tz is None:
        index = index.tz_localize(timezone)
    return index.tz_convert("UTC")


def _at(series: pd.Series, times: np.ndarray, timezone: str) -> np.ndarray:
    return series.reindex(_sequence_index(times, timezone)).to_numpy(dtype=float)


def simple_predictions(series: pd.Series, issue: np.ndarray, target_times: np.ndarray,
                       seasonal_lag: int, windows: list[int], timezone: str) -> dict[str, np.ndarray]:
    step = series.index.to_series().diff().dropna().mode().iloc[0]
    issue_index = _sequence_index(issue, timezone)
    target_index = _sequence_index(target_times, timezone)
    result = {"Naive": _at(series, issue, timezone),
              "SeasonalNaive": series.reindex(target_index - seasonal_lag * step).to_numpy(dtype=float)}
    for window in windows:
        moving = series.rolling(window, min_periods=window).mean()
        result[f"MovingAverage_w{window}"] = moving.reindex(issue_index).to_numpy(dtype=float)
    return result


def longest_segment(series: pd.Series) -> pd.Series:
    valid = series.notna()
    groups = (valid != valid.shift()).cumsum()
    candidates = [part.dropna() for _, part in series.groupby(groups) if valid.loc[part.index[0]]]
    if not candidates:
        raise RuntimeError("Tidak ada segmen suhu lengkap untuk ARIMA.")
    return max(candidates, key=len)


def latest_history(series: pd.Series, issue: pd.Timestamp, step: pd.Timedelta) -> pd.Series:
    """Contiguous observed history ending at issue, never a stale earlier segment."""
    history = series.loc[:issue]
    if history.empty or history.index[-1] != issue or not np.isfinite(history.iloc[-1]):
        return history.iloc[:0]
    finite = np.isfinite(history.to_numpy(dtype=float))
    breaks = np.flatnonzero(~finite)
    start = int(breaks[-1] + 1) if len(breaks) else 0
    # Also preserve calendar gaps if an input omitted missing rows entirely.
    calendar_breaks = np.flatnonzero((history.index[1:] - history.index[:-1]) != step)
    if len(calendar_breaks):
        start = max(start, int(calendar_breaks[-1] + 1))
    return history.iloc[start:]


def select_arima(train: pd.Series, cfg: dict) -> tuple[object, tuple[int, int, int], dict]:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller

    segment = longest_segment(train)
    d, transformed, adf_rows = 0, segment.to_numpy(dtype=float), []
    for candidate_d in range(int(cfg["max_d"]) + 1):
        if len(transformed) < 30:
            break
        statistic, pvalue, *_ = adfuller(transformed, autolag="AIC")
        adf_rows.append({"d": candidate_d, "statistic": float(statistic), "p_value": float(pvalue)})
        d = candidate_d
        if pvalue < 0.05:
            break
        transformed = np.diff(transformed)

    best, best_order, best_aic = None, None, np.inf
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for p in range(int(cfg["max_p"]) + 1):
            for q in range(int(cfg["max_q"]) + 1):
                try:
                    fitted = ARIMA(segment.to_numpy(), order=(p, d, q)).fit()
                    if np.isfinite(fitted.aic) and fitted.aic < best_aic:
                        best, best_order, best_aic = fitted, (p, d, q), float(fitted.aic)
                except Exception:
                    continue
    if best is None:
        raise RuntimeError("Tidak ada kandidat ARIMA yang berhasil diestimasi.")
    return best, best_order, {"adf": adf_rows, "order": list(best_order), "aic": best_aic,
                              "training_segment_start": segment.index[0].isoformat(),
                              "training_segment_end": segment.index[-1].isoformat(),
                              "training_segment_n": len(segment)}


def arima_predictions(fitted: object, full_series: pd.Series, issues: np.ndarray,
                      horizon: int, timezone: str) -> np.ndarray:
    if not full_series.index.is_monotonic_increasing or full_series.index.has_duplicates:
        raise ValueError("ARIMA requires sorted, unique history timestamps.")
    if horizon < 1:
        raise ValueError("ARIMA horizon must be positive.")
    output = np.full(len(issues), np.nan)
    issue_index = _sequence_index(issues, timezone)
    if issue_index.has_duplicates or not issue_index.is_monotonic_increasing:
        raise ValueError("ARIMA issues must be sorted and unique.")
    expected_step = full_series.index.to_series().diff().dropna().mode().iloc[0]
    state, previous_issue = None, None
    for i, issue in enumerate(issue_index):
        try:
            if state is None or previous_issue is None or issue - previous_issue != expected_step:
                history = latest_history(full_series, issue, expected_step)
                if len(history) < 50:
                    continue
                state = fitted.apply(history.to_numpy(dtype=float), refit=False)
            else:
                observation = full_series.reindex([issue]).iloc[0]
                if not np.isfinite(observation):
                    state = None
                    continue
                state = state.append([float(observation)], refit=False)
            output[i] = float(state.forecast(steps=horizon)[-1])
            previous_issue = issue
        except Exception:
            state, previous_issue = None, None
    return output


def choose_moving_average(validation: dict[str, np.ndarray], y_true: np.ndarray) -> tuple[str, list[dict]]:
    rows = []
    for name, prediction in validation.items():
        if name.startswith("MovingAverage_"):
            ok = np.isfinite(prediction) & np.isfinite(y_true)
            score = float(np.mean(np.abs(y_true[ok] - prediction[ok]))) if ok.any() else np.inf
            rows.append({"model": name, "validation_mae": score, "n": int(ok.sum())})
    if not rows:
        raise RuntimeError("Tidak ada Moving Average yang bisa dinilai.")
    rows.sort(key=lambda row: (row["validation_mae"], row["model"]))
    return rows[0]["model"], rows


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="config.yaml")
    parser.add_argument("--run-dir", required=True)
    args = parser.parse_args()
    cfg, run_dir = common.load_config(args.config), Path(args.run_dir).resolve()
    timezone = cfg["project"]["timezone"]
    series = _series(run_dir, cfg["data"]["kolom_target"])
    train_end = pd.Timestamp(cfg["preprocessing"]["split"]["train"][1]).tz_convert("UTC")
    fitted, _, arima_meta = select_arima(series.loc[series.index < train_end], cfg["baseline"]["arima"])
    metadata = {"arima": {**arima_meta,
                           "forecast_history_policy": "latest_contiguous_segment_ending_at_issue",
                           "forecast_minimum_history_slots": 50,
                           "refit_at_issue": False},
                "baseline_code_sha256": common.sha256_file(Path(__file__)), "horizons": {}}

    for horizon in cfg["preprocessing"]["horizons"]:
        seq = common.load_sequence(run_dir, horizon)
        val = simple_predictions(series, seq["issue_val"], seq["t_val"],
                                 cfg["baseline"]["seasonal_naive_lag"], cfg["baseline"]["moving_average_windows"], timezone)
        selected_ma, candidates = choose_moving_average(val, seq["ycel_val"])
        test = simple_predictions(series, seq["issue_test"], seq["t_test"],
                                  cfg["baseline"]["seasonal_naive_lag"], cfg["baseline"]["moving_average_windows"], timezone)
        val["ARIMA"] = arima_predictions(fitted, series, seq["issue_val"], horizon, timezone)
        test["ARIMA"] = arima_predictions(fitted, series, seq["issue_test"], horizon, timezone)
        payload = {"y_true_val": seq["ycel_val"], "issue_val": seq["issue_val"], "t_val": seq["t_val"],
                   "y_true_test": seq["ycel_test"], "issue_test": seq["issue_test"], "t_test": seq["t_test"]}
        for name in ["Naive", "SeasonalNaive", selected_ma, "ARIMA"]:
            public = "MovingAverage" if name == selected_ma else name
            payload[f"{public}_val"] = val[name]
            payload[f"{public}_test"] = test[name]
        np.savez_compressed(run_dir / "results" / f"baseline_predictions_h{horizon}.npz", **payload)
        metadata["horizons"][str(horizon)] = {"minutes": horizon * 5,
                                                "moving_average_selected": selected_ma,
                                                "moving_average_validation": candidates}
    common.atomic_json(run_dir / "results" / "baseline_selection.json", metadata)
    print(json.dumps(metadata, indent=2))


if __name__ == "__main__":
    main()
'''

In [ ]:
# 06_evaluasi/05_evaluate.py
KODE_PIPELINE['06_evaluasi/05_evaluate.py'] = '''"""Final seed-aware evaluation. No seed pooling and no implicit protocol decisions."""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
import sys
from pathlib import Path
PIPELINE_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PIPELINE_ROOT / 'fungsi_bersama'))
sys.path.insert(0, str(PIPELINE_ROOT / '04_training'))
import common
import pc_training_state as state

BASELINES = ('Naive', 'SeasonalNaive', 'MovingAverage', 'ARIMA')
SEEDS = (1, 2, 3)


def prediction_frame(package, baseline=False):
    def timestamp(values):
        parsed = pd.DatetimeIndex(pd.to_datetime(values))
        return (parsed.tz_localize('Asia/Jakarta') if parsed.tz is None else parsed).tz_convert('UTC')
    t_key, y_key, issue_key = ('t_test', 'y_true_test', 'issue_test') if baseline else ('t', 'y_true', 'issue')
    fields = {'target': timestamp(package[t_key]),
              'issue': timestamp(package[issue_key]),
              'y': np.asarray(package[y_key]).reshape(-1)}
    names = BASELINES if baseline else ('lstm', 'gru')
    for name in names:
        fields[name if baseline else name.upper()] = np.asarray(package[f'{name}_test']).reshape(-1)
    frame = pd.DataFrame(fields).set_index('target').sort_index()
    if frame.index.has_duplicates or frame['issue'].duplicated().any():
        raise ValueError('Duplicate target/issue timestamp; evaluation would double-count observations.')
    return frame


def align_packages(baseline, neural, horizon_slots):
    """One intersection for all architectures, baselines and seeds at each horizon."""
    if set(neural) != set(SEEDS):
        raise ValueError('Final evaluation requires all three seeds 1, 2, 3.')
    frames = {'baseline': prediction_frame(baseline, True)}
    frames.update({seed: prediction_frame(package) for seed, package in neural.items()})
    targets = frames['baseline'].index
    for frame in frames.values():
        targets = targets.intersection(frame.index)
    targets = targets.sort_values()
    if targets.empty:
        raise ValueError('No shared target timestamps.')
    reference = frames['baseline'].loc[targets]
    mask = np.isfinite(reference['y'].to_numpy())
    for frame in frames.values():
        frame = frame.loc[targets]
        if not np.allclose(frame['y'], reference['y'], rtol=0, atol=1e-8, equal_nan=True):
            raise ValueError('Ground truth differs at identical target timestamps.')
        if not frame['issue'].equals(reference['issue']):
            raise ValueError('Issue timestamps differ at identical targets.')
        if not ((frame.index - pd.DatetimeIndex(frame['issue'])) == pd.Timedelta(minutes=horizon_slots * 5)).all():
            raise ValueError('Target minus issue does not equal the configured horizon.')
        numeric = frame.select_dtypes(include=[np.number]).to_numpy()
        mask &= np.isfinite(numeric).all(axis=1)
    selected = targets[mask]
    if selected.empty:
        raise ValueError('No finite common intersection across all three seeds and baselines.')
    return {key: frame.loc[selected] for key, frame in frames.items()}, len(targets) - len(selected)


def regression_metrics(y, prediction):
    error = np.asarray(y) - np.asarray(prediction)
    nonzero = np.abs(y) > 1e-8
    return {'n': len(error), 'mae': float(np.abs(error).mean()),
            'rmse': float(np.sqrt(np.square(error).mean())),
            'mape': float((np.abs(error[nonzero] / y[nonzero])).mean() * 100) if nonzero.all() else np.nan,
            'mape_status': 'valid' if nonzero.all() else 'undefined_actual_zero_or_near_zero'}


def seed_summary(rows):
    frame = pd.DataFrame(rows)
    neural = frame[frame['model'].isin(['LSTM', 'GRU'])]
    result = []
    for (model, horizon), group in neural.groupby(['model', 'horizon_minutes']):
        if set(group['seed']) != set(SEEDS) or len(group) != 3 or group['n'].nunique() != 1:
            raise ValueError('Summary requires exactly three seeds on the same intersection.')
        item = {'model': model, 'horizon_minutes': horizon, 'seeds': '1,2,3', 'n_seeds': 3,
                'n_test_per_seed': int(group['n'].iloc[0]), 'sd_ddof': 1}
        for metric in ('mae', 'rmse', 'mape'):
            values = group[metric].to_numpy()
            item[f'{metric}_mean'] = float(values.mean())
            item[f'{metric}_sd'] = float(values.std(ddof=1))
        result.append(item)
    return pd.DataFrame(result)


def dm_hln(y, first, second, horizon):
    differential = np.abs(y - first) - np.abs(y - second)
    n = len(differential)
    if n <= horizon:
        return np.nan, np.nan, 'insufficient_samples'
    centered = differential - differential.mean()
    variance = np.dot(centered, centered) / n
    for lag in range(1, horizon):
        variance += 2 * np.dot(centered[lag:], centered[:-lag]) / n
    if variance <= 0:
        return np.nan, np.nan, 'nonpositive_long_run_variance'
    correction = np.sqrt((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n)
    statistic = differential.mean() / np.sqrt(variance / n) * correction
    return float(statistic), float(2 * student_t.sf(abs(statistic), n - 1)), 'computed'


def calendar_dm_hln(times, y, first, second, horizon):
    """Do not compress missing calendar slots into adjacent DM observations."""
    if len(times) != len(y):
        raise ValueError('DM timestamps and observations must have equal length.')
    if not ((times[1:] - times[:-1]) == pd.Timedelta(minutes=5)).all():
        return np.nan, np.nan, 'gapped_intersection'
    return dm_hln(y, first, second, horizon)


def holm_nine(values):
    """Keep the locked nine-member family even if some tests are undefined."""
    if len(values) != 9:
        raise ValueError('Holm family must contain nine comparisons.')
    values = np.asarray(values, dtype=float)
    order = np.argsort(np.where(np.isfinite(values), values, 1.0))
    output = np.full(9, np.nan)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (9 - rank) * (values[index] if np.isfinite(values[index]) else 1.0))
        if np.isfinite(values[index]):
            output[index] = min(running, 1.0)
    return output


def verify_manifest_artifact(run_dir, relative, digest):
    path = (run_dir / relative).resolve()
    if not path.is_relative_to(run_dir.resolve()) or not path.is_file() or common.sha256_file(path) != digest:
        raise ValueError(f'Artifact path/hash invalid: {relative}')


def evaluate(cfg, run_dir, *, dm_family=None):
    """Evaluate frozen training artifacts with a separately recorded analysis protocol."""
    if dm_family not in (None, "per_seed_per_horizon"):
        raise ValueError("Unsupported analysis DM family.")
    results = run_dir / 'results'
    progress = pd.read_csv(results / 'final_training_progress.csv')
    identity = json.loads((run_dir / 'state' / 'training_identity.json').read_text())
    if common.stable_hash(identity['config']) != common.stable_hash({k: v for k, v in cfg.items() if not k.startswith('_')}):
        raise ValueError('Evaluation config differs from locked training config.')
    fingerprint = identity['fingerprint']
    manifest = json.loads((results / 'final_models_manifest.json').read_text())
    if manifest['run_fingerprint'] != fingerprint:
        raise ValueError('Final manifest fingerprint differs from training identity.')
    predictions_manifest = {item['path']: item for item in manifest['predictions']}
    successful = progress[progress['status'].eq('success') & progress['fingerprint'].eq(fingerprint) & progress['phase'].eq('final')]
    rows, dm_rows, intersection_rows = [], [], []
    dm_enabled = (dm_family or cfg['evaluation'].get('dm_final_seed_family')) == 'per_seed_per_horizon'
    for horizon in cfg['preprocessing']['horizons']:
        for seed in SEEDS:
            for architecture in ('lstm', 'gru'):
                matched = successful[successful['seed'].eq(seed) & successful['horizon_slots'].eq(horizon)
                                     & successful['architecture'].str.lower().eq(architecture)]
                if len(matched) != 1:
                    raise ValueError(f'Missing or duplicate successful final fit: {architecture}, h={horizon}, seed={seed}.')
                row = matched.iloc[0].to_dict()
                state.verify_artifacts(run_dir, row)
                models = [item for item in manifest['models'] if item['architecture'] == architecture and item['horizon_slots'] == horizon and item['seed'] == seed]
                if len(models) != 1 or models[0]['model_sha256'] != row['model_sha256']:
                    raise ValueError('Final model manifest differs from verified fit progress.')
                verify_manifest_artifact(run_dir, models[0]['model_path'], models[0]['model_sha256'])
        with np.load(results / f'baseline_predictions_h{horizon}.npz', allow_pickle=False) as package:
            baseline = dict(package)
        neural = {}
        for seed in SEEDS:
            relative = f'results/neural_predictions_h{horizon}_seed{seed}.npz'
            entry = predictions_manifest.get(relative)
            if entry is None or entry['horizon_slots'] != horizon or entry['seed'] != seed:
                raise ValueError('Prediction artifact missing or mismatched in final manifest.')
            verify_manifest_artifact(run_dir, relative, entry['sha256'])
            with np.load(run_dir / relative, allow_pickle=False) as package:
                neural[seed] = dict(package)
            if str(neural[seed]['run_fingerprint'].item()) != fingerprint or int(neural[seed]['seed'].item()) != seed:
                raise ValueError('Prediction package seed/fingerprint does not match current run.')
        aligned, dropped = align_packages(baseline, neural, horizon)
        reference, times = aligned['baseline'], aligned['baseline'].index
        intersection_rows.append({'horizon_minutes': horizon * 5, 'n': len(times), 'dropped_nonfinite': dropped,
                                  'calendar_gap_count': int(((times[1:] - times[:-1]) != pd.Timedelta(minutes=5)).sum()),
                                  'first_target': times[0], 'last_target': times[-1]})
        y = reference['y'].to_numpy()
        for name in BASELINES:
            rows.append({'model': name, 'horizon_minutes': horizon * 5, 'seed': 0,
                         **regression_metrics(y, reference[name].to_numpy())})
        for seed in SEEDS:
            predictions = {name: reference[name].to_numpy() for name in BASELINES}
            predictions.update({name: aligned[seed][name].to_numpy() for name in ('LSTM', 'GRU')})
            for name in ('LSTM', 'GRU'):
                rows.append({'model': name, 'horizon_minutes': horizon * 5, 'seed': seed,
                             **regression_metrics(y, predictions[name])})
            if dm_enabled:
                family = []
                for first, second in [(a, b) for a in ('LSTM', 'GRU') for b in BASELINES] + [('LSTM', 'GRU')]:
                    # Removing missing timestamps breaks equal-spaced lag assumptions.
                    statistic, p, status = calendar_dm_hln(times, y, predictions[first], predictions[second], horizon)
                    family.append({'seed': seed, 'horizon_minutes': horizon * 5, 'model_1': first, 'model_2': second,
                                   'n': len(y), 'dm_statistic': statistic, 'p_value': p, 'status': status,
                                   'mae_difference_1_minus_2': float((np.abs(y-predictions[first])-np.abs(y-predictions[second])).mean())})
                for item, adjusted in zip(family, holm_nine([item['p_value'] for item in family])):
                    item.update(p_holm=adjusted, reject_h0=bool(np.isfinite(adjusted) and adjusted < cfg['evaluation']['alpha']), family='nine_pairs_per_seed_per_horizon')
                dm_rows.extend(family)
    common.atomic_csv(results / 'metrics_per_seed.csv', pd.DataFrame(rows))
    common.atomic_csv(results / 'metrics_seed_summary.csv', seed_summary(rows))
    common.atomic_csv(results / 'evaluation_intersections.csv', pd.DataFrame(intersection_rows))
    status = {'schema_version': 1, 'seeds': list(SEEDS), 'seed_pooling': False,
              'dm': 'computed_per_seed_per_horizon' if dm_enabled else 'blocked_seed_family_decision_required',
              'dm_limitation': 'Holm controls each nine-pair seed/horizon family, not all seeds jointly. Undefined tests retain family size nine.',
              'early_warning': 'blocked_protocol_decision_required',
              'early_warning_reason': 'Confirm prediction >27 versus >=27; cooldown scope and multiple warnings/event matching. Issue time and actual at issue must be used, never target time as trigger.',
              'timestamp_contract': {
                  'interval': '[bin_start, bin_start + 5 minutes)',
                  'issue': 'start label of the last input bin; not actual issuance time',
                  'input_available_at_nominal': 'issue + 5 minutes; ignores acquisition/processing delay',
                  'target_start': 't = issue + horizon_slots * 5 minutes',
                  'target_end': 't + 5 minutes',
                  'nominal_lead_to_target_end_minutes': 'horizon_slots * 5',
                  'nominal_lead_to_target_start_minutes': '(horizon_slots - 1) * 5',
                  'issued_at_actual': 'not recorded in offline prediction packages; do not infer live lead time'},
              'mape_limitation': 'Celsius has an arbitrary zero; MAPE is retained for historical compatibility, not model selection. Undefined for actual zero/near zero.'}
    if dm_enabled:
        common.atomic_csv(results / 'dm_holm_per_seed.csv', pd.DataFrame(dm_rows))
    common.atomic_json(results / 'evaluation_status.json', status)
    return status


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', default='config.yaml')
    parser.add_argument('--run-dir', required=True)
    args = parser.parse_args()
    print(json.dumps(evaluate(common.load_config(args.config), Path(args.run_dir).resolve()), indent=2))

if __name__ == '__main__':
    main()
'''

In [ ]:
# pemeriksaan_kode/test_baseline_history.py
KODE_PIPELINE['pemeriksaan_kode/test_baseline_history.py'] = '''"""Synthetic regression checks for forecast origin; not thesis performance scores."""
import importlib.util
from pathlib import Path
import unittest

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

ROOT = Path(__file__).resolve().parents[1]
spec = importlib.util.spec_from_file_location('baseline_history', ROOT / '05_baseline/03_baselines.py')
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)


class RecordedFit:
    def __init__(self):
        self.histories, self.appends = [], []

    def apply(self, history, refit):
        assert refit is False
        self.histories.append(history.copy())
        self.last = history[-1]
        return self

    def append(self, values, refit):
        assert refit is False
        self.appends.extend(values)
        self.last = values[-1]
        return self

    def forecast(self, steps):
        return np.repeat(self.last, steps)


def history():
    index = pd.date_range('2026-09-01', periods=210, freq='5min', tz='UTC')
    values = np.arange(210, dtype=float)
    values[140:150] = np.nan  # Old segment is longer than current segment.
    return pd.Series(values, index=index)


class BaselineHistoryTests(unittest.TestCase):
    def test_latest_segment_ends_exactly_at_issue_without_future(self):
        series = history()
        selected = baseline.latest_history(series, series.index[199], pd.Timedelta(minutes=5))
        self.assertEqual(selected.index[0], series.index[150])
        self.assertEqual(selected.index[-1], series.index[199])
        self.assertEqual(len(selected), 50)

    def test_reset_forecast_uses_latest_segment_and_append_next_observation(self):
        series, fit = history(), RecordedFit()
        issues = series.index[[199, 200]].to_numpy()
        prediction = baseline.arima_predictions(fit, series, issues, 12, 'UTC')
        np.testing.assert_array_equal(fit.histories[0], np.arange(150., 200.))
        self.assertEqual(fit.appends, [200.])
        np.testing.assert_array_equal(prediction, [199., 200.])

    def test_real_random_walk_forecast_origin_is_latest_observation(self):
        series = history()
        fitted = ARIMA(series.iloc[:100].to_numpy(), order=(0, 1, 0)).fit()
        prediction = baseline.arima_predictions(fitted, series, series.index[[199, 200]].to_numpy(), 12, 'UTC')
        np.testing.assert_allclose(prediction, [199., 200.], atol=1e-6)

    def test_short_latest_segment_does_not_fall_back_to_stale_long_segment(self):
        series, fit = history(), RecordedFit()
        prediction = baseline.arima_predictions(fit, series, series.index[[190]].to_numpy(), 12, 'UTC')
        self.assertTrue(np.isnan(prediction[0]))
        self.assertEqual(fit.histories, [])

    def test_missing_issue_or_nonfinite_issue_never_uses_older_origin(self):
        series = history()
        for issue in (series.index[145], series.index[199] + pd.Timedelta(minutes=1)):
            self.assertTrue(baseline.latest_history(series, issue, pd.Timedelta(minutes=5)).empty)

    def test_omitted_calendar_rows_are_a_segment_break(self):
        series = history().drop(history().index[180])
        selected = baseline.latest_history(series, series.index[-1], pd.Timedelta(minutes=5))
        self.assertEqual(selected.index[0], pd.Timestamp('2026-09-01 15:05', tz='UTC'))

    def test_duplicate_or_out_of_order_issues_rejected(self):
        series = history()
        for positions in ([200, 200], [200, 199]):
            with self.assertRaisesRegex(ValueError, 'sorted and unique'):
                baseline.arima_predictions(RecordedFit(), series, series.index[positions].to_numpy(), 12, 'UTC')


if __name__ == '__main__':
    unittest.main()
'''

In [ ]:
# pemeriksaan_kode/test_data_identity.py
KODE_PIPELINE['pemeriksaan_kode/test_data_identity.py'] = '''import sys, tempfile, unittest
from pathlib import Path
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT / 'fungsi_bersama'))
import common

class DataIdentityTests(unittest.TestCase):
    def test_missing_columns_rejected_before_preparation(self):
        cfg=common.load_config(ROOT/'config.yaml')
        with tempfile.TemporaryDirectory() as temp:
            source=Path(temp)/'changed.csv'
            source.write_text('waktu_wib,temperature\\n2026-08-20,26\\n')
            cfg['data']['csv_path']=str(source)
            with self.assertRaisesRegex(ValueError,'Kolom dataset belum lengkap'):
                common.prepare_dataset(cfg,Path(temp)/'run')
    def test_existing_run_cannot_be_overwritten(self):
        cfg=common.load_config(ROOT/'config.yaml')
        with tempfile.TemporaryDirectory() as temp:
            source = Path(temp) / 'sensor_raw.csv'
            source.write_text('waktu_wib,temperature,humidity\\n2026-08-20,26,48\\n')
            cfg['data']['csv_path'] = str(source)
            run = Path(temp) / 'run'
            (run / 'dataset').mkdir(parents=True)
            marker=run/'dataset/metadata.json';marker.write_text('{"original":true}')
            with self.assertRaises(FileExistsError):
                common.prepare_dataset(cfg,run)
            self.assertEqual(marker.read_text(),'{"original":true}')

if __name__=='__main__':unittest.main()
'''

In [ ]:
# pemeriksaan_kode/test_final_evaluation.py
KODE_PIPELINE['pemeriksaan_kode/test_final_evaluation.py'] = '''"""Synthetic checks only: alignment, no seed pooling, and protocol gates."""
import importlib.util
import sys
import json
import tempfile
import pandas as pd
import unittest
from pathlib import Path
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'fungsi_bersama'))
sys.path.insert(0, str(ROOT / '04_training'))
spec = importlib.util.spec_from_file_location('final_evaluation', ROOT / '06_evaluasi/05_evaluate.py')
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)


def packages():
    target = np.array(['2026-09-22T01:00', '2026-09-22T01:05', '2026-09-22T01:10'], dtype='datetime64[m]')
    issue = target - np.timedelta64(60, 'm')
    y = np.array([26., 27., 28.])
    baseline = {'t_test': target, 'issue_test': issue, 'y_true_test': y}
    baseline.update({f'{name}_test': y + 1 for name in ev.BASELINES})
    neural = {seed: {'t': target[::-1], 'issue': issue[::-1], 'y_true': y[::-1],
                     'lstm_test': y[::-1] + seed, 'gru_test': y[::-1] - seed} for seed in ev.SEEDS}
    return baseline, neural


class EvaluationTests(unittest.TestCase):
    def test_reorders_timestamp_and_uses_all_seed_intersection(self):
        baseline, neural = packages()
        neural[3]['gru_test'][1] = np.nan
        frames, dropped = ev.align_packages(baseline, neural, 12)
        self.assertEqual(dropped, 1)
        self.assertEqual(len(frames['baseline']), 2)
        np.testing.assert_allclose(frames[1]['LSTM'], frames['baseline']['y'] + 1)
        self.assertEqual(str(frames['baseline'].index[0]), '2026-09-21 18:00:00+00:00')

    def test_missing_seed_rejected(self):
        baseline, neural = packages()
        del neural[2]
        with self.assertRaisesRegex(ValueError, 'three seeds'):
            ev.align_packages(baseline, neural, 12)

    def test_wrong_target_issue_rejected(self):
        baseline, neural = packages()
        neural[1]['issue'] = neural[1]['issue'] - np.timedelta64(5, 'm')
        with self.assertRaisesRegex(ValueError, 'Issue timestamps'):
            ev.align_packages(baseline, neural, 12)

    def test_ground_truth_mismatch_rejected(self):
        baseline, neural = packages()
        neural[1]['y_true'] = neural[1]['y_true'] + .1
        with self.assertRaisesRegex(ValueError, 'Ground truth'):
            ev.align_packages(baseline, neural, 12)

    def test_summary_counts_seeds_not_pooled_observations(self):
        rows = [{'model': model, 'horizon_minutes': 60, 'seed': seed, 'n': 100,
                 'mae': seed, 'rmse': seed, 'mape': seed} for model in ('LSTM', 'GRU') for seed in ev.SEEDS]
        summary = ev.seed_summary(rows)
        self.assertTrue((summary.n_test_per_seed == 100).all())
        self.assertTrue((summary.n_seeds == 3).all())
        self.assertTrue((summary.mae_sd == 1).all())
        with self.assertRaises(ValueError):
            ev.seed_summary(rows[:-1])

    def test_complete_evaluation_writes_counts_and_explicit_gates(self):
        with tempfile.TemporaryDirectory() as folder:
            run = Path(folder)
            for directory in ('results', 'state', 'models'):
                (run / directory).mkdir()
            cfg = {'preprocessing': {'horizons': [12]}, 'evaluation': {'alpha': .05}}
            (run / 'state' / 'training_identity.json').write_text(json.dumps({'fingerprint': 'synthetic', 'config': cfg}))
            baseline, neural = packages()
            np.savez(run / 'results' / 'baseline_predictions_h12.npz', **baseline)
            progress, models, predictions = [], [], []
            for seed, package in neural.items():
                relative = f'results/neural_predictions_h12_seed{seed}.npz'
                np.savez(run / relative, **package, run_fingerprint='synthetic', seed=seed)
                predictions.append({'path': relative, 'sha256': ev.common.sha256_file(run / relative), 'horizon_slots': 12, 'seed': seed})
                for architecture in ('lstm', 'gru'):
                    row = {'status': 'success', 'phase': 'final', 'fingerprint': 'synthetic',
                           'horizon_slots': 12, 'architecture': architecture, 'seed': seed}
                    for prefix in ('model', 'history', 'checkpoint'):
                        artifact = f'models/{architecture}_{seed}_{prefix}.txt'
                        (run / artifact).write_text('synthetic artifact, never real model')
                        row[f'{prefix}_path'] = artifact
                        row[f'{prefix}_sha256'] = ev.common.sha256_file(run / artifact)
                    progress.append(row)
                    models.append({key: row[key] for key in ('architecture', 'horizon_slots', 'seed', 'model_path', 'model_sha256')})
            (run / 'results' / 'final_models_manifest.json').write_text(json.dumps({'run_fingerprint': 'synthetic', 'models': models, 'predictions': predictions}))
            progress_path = run / 'results' / 'final_training_progress.csv'
            pd.DataFrame(progress).to_csv(progress_path, index=False)
            status = ev.evaluate(cfg, run)
            self.assertEqual(status['dm'], 'blocked_seed_family_decision_required')
            self.assertEqual(status['early_warning'], 'blocked_protocol_decision_required')
            metrics = pd.read_csv(run / 'results' / 'metrics_per_seed.csv')
            self.assertEqual(len(metrics), 10)
            self.assertTrue((metrics.n == 3).all())
            summary = pd.read_csv(run / 'results' / 'metrics_seed_summary.csv')
            self.assertEqual(len(summary), 2)
            self.assertTrue((summary.n_test_per_seed == 3).all())
            altered = dict(cfg, evaluation={'alpha': .01})
            with self.assertRaisesRegex(ValueError, 'locked training config'):
                ev.evaluate(altered, run)
            changed_progress = [dict(row, fingerprint='old-run') for row in progress]
            pd.DataFrame(changed_progress).to_csv(progress_path, index=False)
            with self.assertRaisesRegex(ValueError, 'Missing or duplicate successful final fit'):
                ev.evaluate(cfg, run)
            pd.DataFrame(progress).to_csv(progress_path, index=False)
            (run / models[0]['model_path']).write_text('tampered')
            with self.assertRaisesRegex(RuntimeError, 'Checksum'):
                ev.evaluate(cfg, run)

    def test_holm_keeps_nine_comparisons_when_undefined(self):
        adjusted = ev.holm_nine([.001] + [np.nan] * 8)
        self.assertAlmostEqual(adjusted[0], .009)
        self.assertTrue(np.isnan(adjusted[1:]).all())

    def test_dm_rejects_gapped_calendar_instead_of_compressing_time(self):
        times = pd.date_range('2026-09-22', periods=30, freq='5min', tz='UTC').delete(10)
        y = np.arange(len(times), dtype=float)
        statistic, pvalue, status = ev.calendar_dm_hln(times, y, y + 1, y + 2, 12)
        self.assertEqual(status, 'gapped_intersection')
        self.assertTrue(np.isnan(statistic) and np.isnan(pvalue))

    def test_dm_on_regular_calendar_preserves_existing_statistic(self):
        times = pd.date_range('2026-09-22', periods=50, freq='5min', tz='UTC')
        y = np.zeros(50)
        first = np.tile([0., 1.], 25)
        second = np.ones(50)
        expected = ev.dm_hln(y, first, second, 1)
        for unit in ['ns', 'us']:
            self.assertEqual(ev.calendar_dm_hln(times.as_unit(unit), y, first, second, 1), expected)

    def test_dm_length_mismatch_rejected(self):
        times = pd.date_range('2026-09-22', periods=3, freq='5min', tz='UTC')
        with self.assertRaisesRegex(ValueError, 'equal length'):
            ev.calendar_dm_hln(times, np.zeros(2), np.zeros(2), np.zeros(2), 1)

    def test_mape_zero_is_explicit_not_epsilon_substituted(self):
        result = ev.regression_metrics(np.array([0., 1.]), np.array([1., 1.]))
        self.assertTrue(np.isnan(result['mape']))
        self.assertEqual(result['mape_status'], 'undefined_actual_zero_or_near_zero')

if __name__ == '__main__':
    unittest.main()
'''

In [ ]:
# pemeriksaan_kode/test_scaler_boundary.py
KODE_PIPELINE['pemeriksaan_kode/test_scaler_boundary.py'] = '''"""Check that future extremes cannot affect train-only preprocessing."""
import sys
import unittest
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'fungsi_bersama'))
import common


class ScalerBoundaryTests(unittest.TestCase):
    def setUp(self):
        self.frame = pd.DataFrame({'temperature': [10., 20., 100., -50.],
                                   'humidity': [30., 50., 99., 1.]},
                                  index=pd.date_range('2026-01-01', periods=4, freq='1D', tz='Asia/Jakarta'))
        self.pre = {'scaler_fit': 'train_only', 'split': {'train': ['2026-01-01 00:00+07:00', '2026-01-03 00:00+07:00']}}

    def test_future_extremes_do_not_change_train_extrema_or_clip(self):
        lo, hi = common.fit_sensor_scaler(self.frame, list(self.frame), self.pre)
        np.testing.assert_array_equal(lo, [10., 30.])
        np.testing.assert_array_equal(hi, [20., 50.])
        transformed = (self.frame - lo) / (hi - lo)
        self.assertEqual(transformed.temperature.iloc[2], 9.)
        self.assertEqual(transformed.temperature.iloc[3], -6.)
        changed = self.frame.copy(); changed.iloc[2:] *= 100
        other_lo, other_hi = common.fit_sensor_scaler(changed, list(changed), self.pre)
        np.testing.assert_array_equal(other_lo, lo)
        np.testing.assert_array_equal(other_hi, hi)

    def test_invalid_mode_rejected(self):
        self.pre['scaler_fit'] = 'typo'
        with self.assertRaises(ValueError):
            common.fit_sensor_scaler(self.frame, list(self.frame), self.pre)

    def test_legacy_config_remains_global(self):
        self.pre.pop('scaler_fit')
        lo, hi = common.fit_sensor_scaler(self.frame, list(self.frame), self.pre)
        np.testing.assert_array_equal(lo, [-50., 1.])
        np.testing.assert_array_equal(hi, [100., 99.])

    def test_empty_train_rejected(self):
        self.pre['split']['train'] = ['2025-01-01 00:00+07:00', '2025-01-02 00:00+07:00']
        with self.assertRaises(ValueError):
            common.fit_sensor_scaler(self.frame, list(self.frame), self.pre)


if __name__ == '__main__':
    unittest.main()
'''

In [ ]:
# pemeriksaan_kode/test_training_state.py
KODE_PIPELINE['pemeriksaan_kode/test_training_state.py'] = '''import importlib.util
import json
from pathlib import Path
import sys
import tempfile
import unittest
from unittest import mock

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'fungsi_bersama'))
sys.path.insert(0, str(ROOT / '04_training'))
import common
import pc_training_state as state
spec = importlib.util.spec_from_file_location("pc_train", ROOT / "04_training/04_train.py")
train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train)
import pandas as pd


class TrainingStateTests(unittest.TestCase):
    def test_identity_rejects_environment_source_or_prepared_drift(self):
        with tempfile.TemporaryDirectory() as tmp:
            run = Path(tmp)
            (run / 'data').mkdir()
            fixture = run / 'data' / 'scaler.json'
            fixture.write_text('{}')
            cfg = {'model': {'final_seeds': [1, 2, 3]}, '_run_dir': tmp}
            metadata = {'sha256': 'dataset', 'fingerprint': 'preprocessing'}
            env = {'packages': {'tensorflow': '2.18.1'}, 'hostname': 'pc', 'captured_at_utc': 't1'}
            identity = state.bind_run_identity(run, cfg, metadata, env)
            self.assertEqual(identity, state.bind_run_identity(run, cfg, metadata, {**env, 'captured_at_utc': 't2'}))
            for changed in [{**env, 'hostname': 'other'}, {**env, 'packages': {'tensorflow': '2.20.0'}}]:
                with self.assertRaises(RuntimeError):
                    state.bind_run_identity(run, cfg, metadata, changed)
            fixture.write_text('{"changed":true}')
            with self.assertRaises(RuntimeError):
                state.bind_run_identity(run, cfg, metadata, env)

    def test_completed_fit_corruption_is_rejected(self):
        with tempfile.TemporaryDirectory() as tmp:
            run = Path(tmp)
            row = {}
            for prefix in ['model', 'history', 'checkpoint']:
                path = run / f'{prefix}.bin'
                path.write_bytes(b'original')
                row[f'{prefix}_path'] = path.name
                row[f'{prefix}_sha256'] = common.sha256_file(path)
            state.verify_artifacts(run, row)
            (run / 'history.bin').write_bytes(b'tampered')
            with self.assertRaises(RuntimeError):
                state.verify_artifacts(run, row)

    def test_selection_ignores_other_run_and_final_seed(self):
        hp = {'units': 32, 'layers': 1, 'dropout': .1, 'learning_rate': .001, 'batch_size': 32}
        rows = []
        for config_id, value in [('a', 2.0), ('b', 1.0)]:
            for architecture in ['lstm', 'gru']:
                rows.append(dict(hp, config_id=config_id, architecture=architecture, horizon_slots=12,
                                 status='success', seed=1, phase='tuning', fingerprint='current', val_mae_celsius=value))
        rows.append(dict(rows[0], fingerprint='old', val_mae_celsius=.001))
        rows.append(dict(rows[0], seed=3, phase='final', val_mae_celsius=.001))
        cfg = {'model': {'seed_tuning': 1}, 'preprocessing': {'horizons': [12]}}
        selected = train.shared_selection(pd.DataFrame(rows), cfg, 'current')
        self.assertEqual(selected['12']['shared_hyperparameters']['config_id'], 'b')
        self.assertEqual(train.shared_selection(pd.DataFrame(rows), cfg, 'unknown'), {})

    def test_interrupted_fit_restarts_seed_preserving_attempts(self):
        class Model:
            def save(self, path):
                Path(path).write_bytes(b'complete-model')
        with tempfile.TemporaryDirectory() as tmp:
            run = Path(tmp)
            (run / 'results').mkdir()
            cfg = {'model': {'seed_tuning': 1}}
            hp = {'config_id': 'test'}
            progress = run / 'results' / 'progress.csv'
            with mock.patch.object(common, 'load_sequence', return_value={}):
                with mock.patch.object(train, 'train_once', side_effect=KeyboardInterrupt):
                    with self.assertRaises(KeyboardInterrupt):
                        train.run_trial(cfg, run, 'fp', 'gru', 12, hp, progress, seed=3, phase='final')
                first = train.load_progress(progress).iloc[0]
                self.assertEqual(first['status'], 'interrupted')
                def success(*args, **kwargs):
                    self.assertEqual(args[4], 3)
                    directory = kwargs['attempt_dir']
                    (directory / 'history.csv').write_text('epoch,loss,val_loss\\n0,1,1\\n')
                    (directory / 'best.keras').write_bytes(b'checkpoint')
                    return Model(), {'val_mae_celsius': 1, 'duration_seconds': .2, 'epochs_ran': 1}
                with mock.patch.object(train, 'train_once', side_effect=success):
                    _, row = train.run_trial(cfg, run, 'fp', 'gru', 12, hp, progress, seed=3, phase='final')
                self.assertNotEqual(first['attempt_dir'], row['attempt_dir'])
                self.assertTrue((run / first['attempt_dir'] / 'status.json').exists())
                with mock.patch.object(train, 'train_once', side_effect=AssertionError('must skip')):
                    model, resumed = train.run_trial(cfg, run, 'fp', 'gru', 12, hp, progress, seed=3, phase='final')
                self.assertIsNone(model)
                self.assertEqual(resumed['seed'], 3)

    def test_final_writes_all_eighteen_fits_and_seed_predictions(self):
        import numpy as np
        from types import SimpleNamespace
        class Model:
            def save(self, path):
                Path(path).write_bytes(b'fixture-not-real-keras')
            def predict(self, x, **kwargs):
                return np.zeros((len(x), 1))
        with tempfile.TemporaryDirectory() as tmp:
            run = Path(tmp)
            for name in ['results', 'models', 'data', 'state']:
                (run / name).mkdir()
            common.atomic_json(run / 'data' / 'scaler.json', {'min': [20], 'max': [40]})
            cfg = {'model': {'seed_tuning': 1, 'final_seeds': [1, 2, 3],
                             'architectures': ['lstm', 'gru']},
                   'preprocessing': {'horizons': [12, 24, 36]}}
            selection = {'run_fingerprint': 'fp', 'selections': {
                str(h): {'shared_hyperparameters': {'config_id': 'fixture'}} for h in [12, 24, 36]}}
            common.atomic_json(run / 'results' / 'selected_shared_configs.json', selection)
            sequence = {'X_val': np.zeros((2, 3, 1)), 'X_test': np.zeros((2, 3, 1)),
                        'ycel_val': np.array([20., 20.]), 'ycel_test': np.array([20., 20.])}
            for split in ['val', 'test']:
                sequence['t_' + split] = np.array(['2026-09-22T01:00', '2026-09-22T01:05'], dtype='datetime64[m]')
                sequence['issue_' + split] = np.array(['2026-09-22T00:00', '2026-09-22T00:05'], dtype='datetime64[m]')
                sequence['actual_at_issue_' + split] = np.array([19., 19.])
            def success(*args, **kwargs):
                directory = kwargs['attempt_dir']
                (directory / 'history.csv').write_text('epoch,loss,val_loss\\n0,1,1\\n')
                (directory / 'best.keras').write_bytes(b'checkpoint')
                return Model(), {'val_mae_celsius': 0, 'duration_seconds': .2, 'epochs_ran': 1}
            fake_tf = SimpleNamespace(keras=SimpleNamespace(backend=SimpleNamespace(clear_session=lambda: None)))
            with mock.patch.dict(sys.modules, {'tensorflow': fake_tf}):
                with mock.patch.object(common, 'load_sequence', return_value=sequence):
                    with mock.patch.object(train, 'train_once', side_effect=success) as fit:
                        train.final_training(cfg, run, 'fp')
                        self.assertEqual(fit.call_count, 18)
            manifest = json.loads((run / 'results' / 'final_models_manifest.json').read_text())
            self.assertEqual(len(manifest['models']), 18)
            self.assertEqual(len(manifest['predictions']), 9)
            frame = train.load_progress(run / 'results' / 'final_training_progress.csv')
            self.assertEqual(set(frame.seed), {1, 2, 3})
            self.assertEqual(set(frame.status), {'success'})
            for item in manifest['predictions']:
                path = run / item['path']
                self.assertEqual(common.sha256_file(path), item['sha256'])
                with np.load(path, allow_pickle=False) as payload:
                    self.assertEqual(int(payload['seed']), item['seed'])
                    self.assertIn('actual_at_issue_test', payload.files)
                    self.assertEqual(str(payload['run_fingerprint']), 'fp')


if __name__ == '__main__':
    unittest.main()
'''

## CSV sumber bawaan

Data sumber disimpan dalam bentuk terkompresi di sel berikut. Notebook memeriksa checksum setelah membukanya kembali.

In [ ]:
COMMIT_SUMBER = '3b07a610e8954d5d95feddc21adc1aa5eff91feb'
DATA_CSV_SHA256 = 'fc7bb35cd04f723d76c1d98c36b063e4b92650291c755f612a2f4690605abfec'
DATA_CSV_B64 = (
    '...'
    '+EF2NG+Z9ZPDpULi0Nipc97uCf4/Cn92iIQyNwA='
)

## Jalankan tahap yang dipilih

In [ ]:
RUN_DIR = mulai()